# Met-3DNet-VI v0.7 — Fine-tuning Run
**Warm start from:** `best_model_v7.pt` (val_auc=0.7959, epoch=44, Focal Loss)

**Goal:** Continue training from checkpoint with fine-tuning LR (1e-4) to push AUROC above 0.81

**Changes from scratch run:**
- LR: 5e-4 → 1e-4 (fine-tuning)
- Warmup: 8 epochs → 3 epochs (weights already oriented)
- Epochs: 100 → 60 (already trained 66+ epochs)
- Checkpoint: best_model_v7.pt loaded at Cell 7 start
**Datasets:** IEDB v3 + TumorAgDB1.0 + TESLA (Wells et al., 2020) + NCI/HiTIDE (Müller et al., 2023)

**Key changes vs v0.6:**
- Added TESLA: 41 IFN-γ ELISpot + multimer validated immunogenic neo-peptides (Wells et al., 2020)
- Added NCI+HiTIDE: 176 IFN-γ ELISpot validated CD8+ T cell immunogenic neo-peptides (Müller et al., 2023)
- Scratch training with correct pos_weight (no warm-start distribution mismatch)
- lambda2=0.0 (functional CE disabled — r(lambda2, AUC)=-0.91 in v0.4)
- Total immunogenic labels: ~10,700 | New IFN-γ validated additions: +217

## Cell 1 — Setup

In [1]:
import os, sys, glob, shutil, subprocess, re
import warnings; warnings.filterwarnings("ignore")

KAGGLE   = os.path.exists("/kaggle/input")
WORK_DIR = "/kaggle/working" if KAGGLE else os.getcwd()
MODELS_DIR = os.path.join(WORK_DIR, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

subprocess.run(["pip","install","pyarrow","torch-geometric",
                "scikit-learn","openpyxl","--quiet"], check=True)

import torch, numpy as np, pandas as pd

def _safe_device():
    """
    Kaggle sometimes assigns a GPU whose compute capability doesn't match
    the installed PyTorch CUDA kernels (cudaErrorNoKernelImageForDevice).
    Probe with a tiny tensor before committing to CUDA.
    """
    if not torch.cuda.is_available():
        return torch.device("cpu")
    try:
        _t = torch.zeros(4, device="cuda")
        _t2 = torch.nn.LayerNorm(4).cuda()(_t)   # LayerNorm is the first op that crashed
        del _t, _t2
        torch.cuda.empty_cache()
        return torch.device("cuda")
    except Exception as _e:
        print(f"  GPU probe failed ({type(_e).__name__}: {_e})")
        print("  Falling back to CPU — training will be slower but correct")
        torch.cuda.empty_cache()
        return torch.device("cpu")

device = _safe_device()
print(f"Device: {device}")
if device.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}  |  CUDA {torch.version.cuda}  |  PyTorch {torch.__version__}")
else:
    print(f"  PyTorch {torch.__version__} (CPU mode)")

INPUT_ROOTS = [
    "/kaggle/input/tesla-nci-labels",  # TESLA + NCI labels
    "/kaggle/input/tesla-labels",       # TESLA (fallback)
    "/kaggle/working/data",              # local uploads
    ]
if KAGGLE:
    for ds in os.listdir("/kaggle/input"):
        INPUT_ROOTS.append(os.path.join("/kaggle/input", ds))

def find_file(name, roots=None):
    """Find a file by name across all dataset roots (recursive)."""
    search_roots = roots or INPUT_ROOTS or [WORK_DIR]
    for root in search_roots:
        for p in glob.glob(os.path.join(root, "**", name), recursive=True):
            if os.path.exists(p):
                return p
    return None

# Copy best_model.pt — recursive search so nested paths work
ckpt_dst = os.path.join(MODELS_DIR, "best_model.pt")
if not os.path.exists(ckpt_dst):
    hit = find_file("best_model.pt")
    if hit:
        shutil.copy2(hit, ckpt_dst)
        print(f"Copied best_model.pt from: {hit}")
    else:
        print("WARNING: best_model.pt not found — training will start from scratch.")
        print("  Add your Kaggle dataset containing best_model.pt to enable warm start.")
else:
    print(f"Checkpoint ready: {ckpt_dst}")

print(f"Setup complete | {len(INPUT_ROOTS)} dataset(s) mounted")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 796.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 5.6 MB/s eta 0:00:00
Device: cuda
  GPU: Tesla T4  |  CUDA 12.8  |  PyTorch 2.10.0+cu128
Copied best_model.pt from: /kaggle/input/datasets/neetuaashi/iedb-org-database-export/best_model.pt
Setup complete | 4 dataset(s) mounted


## Cell 2 — Audit TumorAgDB1.0 Files
Reads every xlsx file, prints its sheets and columns, identifies peptide + HLA + label columns.
Run this first to understand the data before extraction.

In [2]:
from openpyxl import load_workbook

# All TumorAgDB1.0 file names (from the dataset description)
XLSX_NAMES = [
    "All the data on T cell activation experiments.xlsx",
    "NeoAntigen-PubData 2024-2025.xlsx",
    "Non-immunogenic Mutation Dataset.xlsx",
    "T-mTSA-negative-mouse.xlsx",
    "T-mTSA-postive-Homo sapiens-812.xlsx",
    "T-mTSA-postive-mouse.xlsx",
    "Tumor Protein Database.xlsx",
    "Validated Immunogenic Neoantigen Data.xlsx",
    "immunogenic Mutation Dataset.xlsx",
    "immunogenic Neo-peptide Dataset.xlsx",
    "mouse-MHC I-MB49-B16F10-P815-BBN963.xlsx",
    "mouse-MHC II-MB49-B16F10.xlsx",
]

# Keywords to look for in column names
PEP_KEYS  = ["peptide","sequence","epitope","neo","mutation","mutant","antigen"]
HLA_KEYS  = ["hla","mhc","allele","restriction","locus"]
LABEL_KEYS= ["immuno","response","positive","negative","label","class",
              "activation","ifn","elispot","validated","functional"]

def audit_xlsx(path):
    """Read xlsx and return sheet summary + column classifications."""
    wb = load_workbook(path, read_only=True, data_only=True)
    results = {}
    for sheet in wb.sheetnames:
        ws = wb[sheet]
        rows = list(ws.iter_rows(max_row=3, values_only=True))
        if not rows: continue
        header = [str(c).strip() if c else "" for c in rows[0]]
        sample  = rows[1] if len(rows) > 1 else []

        pep_cols   = [h for h in header if any(k in h.lower() for k in PEP_KEYS)]
        hla_cols   = [h for h in header if any(k in h.lower() for k in HLA_KEYS)]
        label_cols = [h for h in header if any(k in h.lower() for k in LABEL_KEYS)]

        results[sheet] = {
            "n_cols"     : len(header),
            "pep_cols"   : pep_cols[:5],
            "hla_cols"   : hla_cols[:5],
            "label_cols" : label_cols[:5],
            "header"     : header[:12],
            "sample"     : [str(v)[:40] if v else "" for v in (sample or [])[:8]],
        }
    wb.close()
    return results

print("Auditing TumorAgDB1.0 files...")
print("="*70)

AUDIT = {}
for fname in XLSX_NAMES:
    path = find_file(fname)
    if path is None:
        print(f"  NOT FOUND: {fname}")
        continue

    sz = os.path.getsize(path) / 1e6
    try:
        info = audit_xlsx(path)
        AUDIT[fname] = {"path": path, "size_mb": sz, "sheets": info}

        # Score usefulness
        has_pep   = any(s["pep_cols"]   for s in info.values())
        has_hla   = any(s["hla_cols"]   for s in info.values())
        has_label = any(s["label_cols"] for s in info.values())
        usability = sum([has_pep, has_hla, has_label])
        flag = "HIGH" if usability == 3 else "MED" if usability >= 2 else "LOW"

        print(f"\n[{flag}] {fname}  ({sz:.1f} MB)")
        for sheet, s in info.items():
            print(f"  Sheet: {sheet}")
            print(f"    Peptide cols : {s['pep_cols'] or 'none found'}")
            print(f"    HLA cols     : {s['hla_cols'] or 'none found'}")
            print(f"    Label cols   : {s['label_cols'] or 'none found'}")
            print(f"    All cols     : {s['header']}")
    except Exception as e:
        print(f"  ERROR reading {fname}: {e}")

print("\nAudit complete ✓")

# ─── TESLA integration (Wells et al., 2020; Cell 183:818-834) ───────────────
def load_tesla(roots):
    # Priority 1: pre-built label CSVs
    csv_names = ["tesla_immunogenicity_labels.csv", "tesla_labels.csv",
                 "TESLA_data.csv", "tesla_neoantigens.csv",
                 "TableS2.csv", "Table_S2.csv", "wells2020_s2.csv"]
    for root in roots:
        for name in csv_names:
            for p in glob.glob(os.path.join(root, "**", name), recursive=True):
                if os.path.exists(p):
                    try:
                        df = pd.read_csv(p)
                        df.columns = [c.strip().lower() for c in df.columns]
                        # Must have peptide + immunogenicity columns
                        pep_c = next((c for c in df.columns if "peptide" in c or "epitope" in c or "sequence" in c), None)
                        lbl_c = next((c for c in df.columns if "immuno" in c or "response" in c or "elispot" in c), None)
                        if pep_c and lbl_c:
                            df = df.rename(columns={pep_c: "peptide", lbl_c: "immunogenicity"})
                            if "hla_allele" not in df.columns:
                                hla_c = next((c for c in df.columns if "hla" in c or "allele" in c or "mhc" in c), None)
                                df["hla_allele"] = df[hla_c].str.strip() if hla_c else "HLA-A*02:01"
                            df["immunogenicity"] = pd.to_numeric(df["immunogenicity"], errors="coerce").fillna(0).astype(int)
                            df = df[df["peptide"].apply(lambda s: valid_pep(str(s).strip().upper()))]
                            print(f"  TESLA loaded: {len(df)} rows from {p}")
                            return df
                    except Exception as e:
                        print(f"  TESLA candidate {p}: {e}")
    # Priority 2: extract from NeoAntigen-PubData xlsx (contains TESLA subset)
    pubdata_path = find_file("NeoAntigen-PubData 2024-2025.xlsx")
    if pubdata_path:
        try:
            xl = pd.ExcelFile(pubdata_path, engine="openpyxl")
            for sheet in xl.sheet_names:
                if "tesla" in sheet.lower() or "wells" in sheet.lower():
                    df = xl.parse(sheet)
                    df.columns = [str(c).strip().lower() for c in df.columns]
                    pep_c = next((c for c in df.columns if "peptide" in c), None)
                    lbl_c = next((c for c in df.columns if "immuno" in c), None)
                    if pep_c and lbl_c:
                        df = df.rename(columns={pep_c: "peptide", lbl_c: "immunogenicity"})
                        if "hla_allele" not in df.columns:
                            df["hla_allele"] = "HLA-A*02:01"
                        df["immunogenicity"] = pd.to_numeric(df["immunogenicity"], errors="coerce").fillna(0).astype(int)
                        print(f"  TESLA extracted from PubData sheet '{sheet}': {len(df)} rows")
                        return df
        except Exception as e:
            print(f"  PubData TESLA extraction failed: {e}")
    print("  WARNING: TESLA labels not found — skipping")
    print("  Add tesla_immunogenicity_labels.csv to your Kaggle dataset (neetuaashi/tesla-nci-labels)")
    return None

# ─── NCI/HiTIDE integration (Müller et al., 2023; Immunity 56:2650) ─────────
def load_nci(roots):
    # Priority 1: pre-built label CSVs
    csv_names = ["nci_muller2023_labels.csv", "nci_hitide_labels.csv",
                 "nci_hitide.csv", "hitide_labels.csv", "muller_2023.csv",
                 "NCI_HiTIDE_labels.csv", "nci_labels.csv"]
    for root in roots:
        for name in csv_names:
            for p in glob.glob(os.path.join(root, "**", name), recursive=True):
                if os.path.exists(p):
                    try:
                        df = pd.read_csv(p)
                        df.columns = [c.strip().lower() for c in df.columns]
                        pep_c = next((c for c in df.columns if "peptide" in c or "epitope" in c), None)
                        lbl_c = next((c for c in df.columns if "immuno" in c or "response" in c), None)
                        if pep_c and lbl_c:
                            df = df.rename(columns={pep_c: "peptide", lbl_c: "immunogenicity"})
                            if "hla_allele" not in df.columns:
                                hla_c = next((c for c in df.columns if "hla" in c or "allele" in c or "mhc" in c), None)
                                df["hla_allele"] = df[hla_c].str.strip() if hla_c else "HLA-A*02:01"
                            df["immunogenicity"] = pd.to_numeric(df["immunogenicity"], errors="coerce").fillna(0).astype(int)
                            df = df[df["peptide"].apply(lambda s: valid_pep(str(s).strip().upper()))]
                            print(f"  NCI/HiTIDE loaded: {len(df)} rows from {p}")
                            return df
                    except Exception as e:
                        print(f"  NCI candidate {p}: {e}")
    # Priority 2: extract from NeoAntigen-PubData xlsx
    pubdata_path = find_file("NeoAntigen-PubData 2024-2025.xlsx")
    if pubdata_path:
        try:
            xl = pd.ExcelFile(pubdata_path, engine="openpyxl")
            for sheet in xl.sheet_names:
                if any(k in sheet.lower() for k in ["nci", "hitide", "muller", "hla-a"]):
                    df = xl.parse(sheet)
                    df.columns = [str(c).strip().lower() for c in df.columns]
                    pep_c = next((c for c in df.columns if "peptide" in c), None)
                    lbl_c = next((c for c in df.columns if "immuno" in c), None)
                    if pep_c and lbl_c:
                        df = df.rename(columns={pep_c: "peptide", lbl_c: "immunogenicity"})
                        if "hla_allele" not in df.columns:
                            df["hla_allele"] = "HLA-A*02:01"
                        df["immunogenicity"] = pd.to_numeric(df["immunogenicity"], errors="coerce").fillna(0).astype(int)
                        print(f"  NCI extracted from PubData sheet '{sheet}': {len(df)} rows")
                        return df
            # Last resort: load the full PubData file as NCI surrogate
            df_all = xl.parse(xl.sheet_names[0])
            df_all.columns = [str(c).strip().lower() for c in df_all.columns]
            pep_c = next((c for c in df_all.columns if "peptide" in c or "sequence" in c), None)
            lbl_c = next((c for c in df_all.columns if "immuno" in c or "response" in c), None)
            if pep_c and lbl_c:
                df_all = df_all.rename(columns={pep_c: "peptide", lbl_c: "immunogenicity"})
                if "hla_allele" not in df_all.columns:
                    hla_c = next((c for c in df_all.columns if "hla" in c or "allele" in c), None)
                    df_all["hla_allele"] = df_all[hla_c].str.strip() if hla_c else "HLA-A*02:01"
                df_all["immunogenicity"] = pd.to_numeric(df_all["immunogenicity"], errors="coerce").fillna(0).astype(int)
                df_all = df_all[df_all["peptide"].apply(lambda s: valid_pep(str(s).strip().upper()))]
                print(f"  NCI surrogate from NeoAntigen-PubData: {len(df_all)} rows")
                return df_all
        except Exception as e:
            print(f"  PubData NCI extraction failed: {e}")
    print("  WARNING: NCI labels not found — skipping")
    print("  Add nci_muller2023_labels.csv to your Kaggle dataset (neetuaashi/tesla-nci-labels)")
    return None

def standardise_external(df, source_name):
    """Convert TESLA/NCI CSV to standard parquet schema for merging."""
    out = pd.DataFrame()
    out["peptide"]          = df["peptide"].str.strip().str.upper()
    out["hla_allele"]       = df["hla_allele"].str.strip()
    out["immunogenicity"]   = df["immunogenicity"].astype(int)
    out["functional_class"] = df.get("functional_class", df["immunogenicity"].apply(lambda x: 2 if x==1 else 0))
    out["source"]           = source_name
    # Keep length filter consistent
    out["seq_len"]          = out["peptide"].str.len()
    out = out[(out["seq_len"] >= 8) & (out["seq_len"] <= 14)]
    out = out.drop(columns=["seq_len"])
    out = out.dropna(subset=["peptide","hla_allele"])
    out = out.drop_duplicates(subset=["peptide","hla_allele"])
    return out

print("\nLoading external validation datasets...")
tesla_df = load_tesla(INPUT_ROOTS + ["/mnt/user-data/uploads", "/kaggle/working"])
nci_df   = load_nci(INPUT_ROOTS + ["/mnt/user-data/uploads", "/kaggle/working"])


Auditing TumorAgDB1.0 files...

[HIGH] All the data on T cell activation experiments.xlsx  (13.8 MB)
  Sheet: Sheet1
    Peptide cols : ['Antigen_Name', 'Epitope_Antigen_IRI', 'Epitope_Organism', 'Length_of_Peptide', 'Peptide']
    HLA cols     : ['MHC_Allele', 'MHC_Type']
    Label cols   : ['immunogenicity']
    All cols     : ['Antigen_Name', 'Assay_Group', 'Assay_Units', 'Epitope_Antigen_IRI', 'Epitope_Organism', 'Length_of_Peptide', 'MHC_Allele', 'MHC_Type', 'Peptide', 'Protein_IRI', 'Qualitative_Measure', 'Reference']

[LOW] NeoAntigen-PubData 2024-2025.xlsx  (0.8 MB)

[HIGH] Non-immunogenic Mutation Dataset.xlsx  (5.7 MB)
  Sheet: Sheet1
    Peptide cols : ['aa_mutant', 'mutant_seq', 'mutation_type', 'nb_same_mutation_Intogen', 'nb_mutations_in_gene_Intogen']
    HLA cols     : ['COUNT_MUT_RANK_CI_MIXMHC', 'COUNT_MUT_RANK_CI_netMHCpan', 'MIN_MUT_RANK_CI_MIXMHC', 'WT_BEST_RANK_CI_MIXMHC']
    Label cols   : ['response_type', 'immunogenicity']
    All cols     : ['dataset', 'respo

## Cell 2b — Build TESLA + NCI label CSVs
Extracts TESLA (Wells et al. 2020, ~41 peptides) and NCI/HiTIDE (Müller et al. 2023, ~176 peptides) from `NeoAntigen-PubData 2024-2025.xlsx`.

**Run once**, then download the output CSVs and add them to the `neetuaashi/tesla-nci-labels` dataset so they persist across sessions.

In [3]:
# ── Build tesla_immunogenicity_labels.csv + nci_muller2023_labels.csv ────────
# Extracts TESLA (Wells 2020, n=41) and NCI/HiTIDE (Müller 2023, n=176)
# from NeoAntigen-PubData 2024-2025.xlsx which is already mounted.
# Saves to /kaggle/working/ so load_tesla/load_nci can find them.
# Run this cell once; add outputs to neetuaashi/tesla-nci-labels dataset
# so they persist across sessions without re-running.

import os, re
import pandas as pd

VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")
HLA_RE   = re.compile(r"HLA-[ABC]\*\d{2}:\d{2}")

def valid_pep_b(s, mn=8, mx=14):
    s = str(s).strip().upper()
    return mn <= len(s) <= mx and all(c in VALID_AA for c in s)

def norm_hla_b(s):
    if not isinstance(s, str): return None
    m = HLA_RE.search(s.strip())
    return m.group() if m else None

def extract_label_csv(xl, sheet_keywords, source_tag, out_name):
    for sheet in xl.sheet_names:
        if not any(k in sheet.lower() for k in sheet_keywords):
            continue
        try:
            df = xl.parse(sheet)
            df.columns = [str(c).strip() for c in df.columns]
            pep_c = next((c for c in df.columns if "Peptide" in c or "peptide" in c
                          or "Epitope" in c or "Sequence" in c), None)
            lbl_c = next((c for c in df.columns if "immunogen" in c.lower()
                          or "response" in c.lower() or "elispot" in c.lower()), None)
            hla_c = next((c for c in df.columns if "MHC_Allele" in c or "HLA" in c
                          or "allele" in c.lower()), None)
            if not pep_c:
                continue
            out_rows = []
            for _, row in df.iterrows():
                pep = str(row.get(pep_c, "")).strip().upper()
                if not valid_pep_b(pep): continue
                lbl = 1 if str(row.get(lbl_c, "0")).strip().lower() in (
                    "1","positive","yes","immunogenic","true") else 0
                hla = norm_hla_b(str(row.get(hla_c, ""))) if hla_c else None
                hla = hla or "HLA-A*02:01"
                out_rows.append({"peptide": pep, "hla_allele": hla,
                                 "immunogenicity": lbl,
                                 "functional_class": 2 if lbl else 0,
                                 "source": source_tag})
            if out_rows:
                out_df = pd.DataFrame(out_rows).drop_duplicates(subset=["peptide","hla_allele"])
                out_path = os.path.join(WORK_DIR, out_name)
                out_df.to_csv(out_path, index=False)
                print(f"  {source_tag}: {len(out_df)} records -> {out_path}")
                print(f"    immunogenic={out_df['immunogenicity'].sum()}  non-immunogenic={(out_df['immunogenicity']==0).sum()}")
                return out_df
        except Exception as e:
            print(f"  {sheet} error: {e}")
    return None

pubdata_path = find_file("NeoAntigen-PubData 2024-2025.xlsx")
if pubdata_path:
    print(f"NeoAntigen-PubData found: {pubdata_path}")
    xl = pd.ExcelFile(pubdata_path, engine="openpyxl")
    print(f"Sheets: {xl.sheet_names}")

    # Try TESLA sheet
    tesla_df_built = extract_label_csv(xl, ["tesla","wells"], "TESLA", "tesla_immunogenicity_labels.csv")

    # Try NCI sheet
    nci_df_built = extract_label_csv(xl, ["nci","hitide","muller"], "NCI_HiTIDE", "nci_muller2023_labels.csv")

    # If no named sheets found, parse all sheets and label by size
    if tesla_df_built is None and nci_df_built is None:
        print("  No TESLA/NCI-named sheets found — parsing all sheets for peptide data")
        all_pep_sheets = []
        for sheet in xl.sheet_names:
            try:
                df = xl.parse(sheet)
                df.columns = [str(c).strip() for c in df.columns]
                pep_c = next((c for c in df.columns if c in
                              ["Peptide","peptide","Epitope","Sequence"]), None)
                lbl_c = next((c for c in df.columns if "immunogen" in c.lower()
                              or "response" in c.lower()), None)
                if pep_c and lbl_c:
                    valid_rows = df[df[pep_c].apply(lambda s: valid_pep_b(str(s)))].copy()
                    all_pep_sheets.append((sheet, len(valid_rows), df, pep_c, lbl_c))
                    print(f"    Sheet '{sheet}': {len(valid_rows)} valid peptide rows")
            except Exception: pass

        # Heuristic: TESLA n≈41, NCI n≈176 — assign by size
        if all_pep_sheets:
            all_pep_sheets.sort(key=lambda x: x[1])
            for sheet, n, df, pep_c, lbl_c in all_pep_sheets:
                hla_c = next((c for c in df.columns if "MHC_Allele" in c or "HLA" in c), None)
                tag   = "TESLA" if n < 100 else "NCI_HiTIDE"
                fname = "tesla_immunogenicity_labels.csv" if n < 100 else "nci_muller2023_labels.csv"
                rows = []
                for _, row in df.iterrows():
                    pep = str(row.get(pep_c, "")).strip().upper()
                    if not valid_pep_b(pep): continue
                    lbl = 1 if str(row.get(lbl_c, "0")).strip().lower() in (
                        "1","positive","yes","immunogenic","true") else 0
                    hla = norm_hla_b(str(row.get(hla_c, ""))) if hla_c else None
                    rows.append({"peptide": pep, "hla_allele": hla or "HLA-A*02:01",
                                 "immunogenicity": lbl, "functional_class": 2 if lbl else 0,
                                 "source": tag})
                if rows:
                    out_df = pd.DataFrame(rows).drop_duplicates(subset=["peptide","hla_allele"])
                    out_path = os.path.join(WORK_DIR, fname)
                    out_df.to_csv(out_path, index=False)
                    print(f"  {tag} (heuristic, n={len(out_df)}) -> {out_path}")
else:
    print("NeoAntigen-PubData 2024-2025.xlsx not found")
    print("Upload it to the neetuaashi/tumoragdb1-0 dataset OR")
    print("create tesla_immunogenicity_labels.csv and nci_muller2023_labels.csv manually.")
    print()
    print("Required CSV format (both files):")
    print("  peptide, hla_allele, immunogenicity, functional_class")
    print("  MMCVDLIAG, HLA-A*02:01, 1, 2")

print("\nAfter building these CSVs:")
print("  Output tab -> download tesla_immunogenicity_labels.csv + nci_muller2023_labels.csv")
print("  Add both to neetuaashi/tesla-nci-labels dataset")
print("  They will be found automatically on future runs")


NeoAntigen-PubData found: /kaggle/input/datasets/neetuaashi/tumoragdb1-0/NeoAntigen-PubData 2024-2025.xlsx
Sheets: []
  No TESLA/NCI-named sheets found — parsing all sheets for peptide data

After building these CSVs:
  Output tab -> download tesla_immunogenicity_labels.csv + nci_muller2023_labels.csv
  Add both to neetuaashi/tesla-nci-labels dataset
  They will be found automatically on future runs


## Cell 3 — Extract Functional Labels from Each File
After reviewing the audit output above, edit the column mappings below if needed.
The defaults are best guesses based on TumorAgDB1.0 standard schema.

In [4]:
VALID_AA    = set("ACDEFGHIKLMNPQRSTVWY")
HLA_PATTERN = re.compile(r"HLA-[ABC]\*\d{2}:\d{2}")

def valid_pep(seq, mn=8, mx=14):
    if not isinstance(seq, str): return False
    s = seq.strip().upper()
    return mn <= len(s) <= mx and all(c in VALID_AA for c in s)

def norm_hla(s):
    if not isinstance(s, str): return None
    s = s.strip()
    m = HLA_PATTERN.search(s)
    if m: return m.group()
    fixed = re.sub(r"^(A|B|C)(\*?)(\d{2})(\:?)(\d{2})$",
                   lambda m: f"HLA-{m.group(1)}*{m.group(3)}:{m.group(5)}", s)
    m = HLA_PATTERN.search(fixed)
    return m.group() if m else None

def find_col_exact(df, priority_names):
    """
    Exact name priority lookup (case-insensitive).
    Do NOT use substring matching — 'peptide' as a substring
    matches 'Length_of_Peptide' before 'Peptide', which was the
    bug causing zero records in every previous extraction attempt.
    """
    col_map = {c.lower(): c for c in df.columns}
    for name in priority_names:
        if name.lower() in col_map:
            return col_map[name.lower()]
    return None

def extract_standard(path, functional_class, source_tag):
    """
    Standard TumorAgDB schema: Peptide + MHC_Allele + immunogenicity.
    Files: T-mTSA-positive-812, Validated Immunogenic, T cell activation.
    """
    records = []
    try:
        xl = pd.ExcelFile(path, engine="openpyxl")
    except Exception as e:
        print(f"    Cannot open: {e}"); return pd.DataFrame()

    for sheet in xl.sheet_names:
        try: df = xl.parse(sheet)
        except Exception: continue
        if len(df) == 0: continue

        pep_col = find_col_exact(df, ["Peptide", "Epitope", "peptide", "epitope"])
        hla_col = find_col_exact(df, ["MHC_Allele", "HLA_Allele", "MHC Allele",
                                       "Allele", "allele"])
        lbl_col = find_col_exact(df, ["immunogenicity", "Qualitative_Measure",
                                       "response_type", "ELISpot_score"])
        if pep_col is None:
            print(f"    [{sheet}] no peptide column — skip"); continue

        n0 = len(records)
        for _, row in df.iterrows():
            pep = str(row.get(pep_col, "")).strip().upper()
            if not valid_pep(pep): continue
            if lbl_col is not None:
                val = str(row.get(lbl_col, "")).strip().lower()
                if functional_class == 2 and val in ("0","negative","non-immunogenic","no"): continue
                if functional_class == 0 and val in ("1","positive","immunogenic","yes"): continue
            hla = None
            if hla_col: hla = norm_hla(str(row.get(hla_col, "")))
            if hla is None:
                for col in df.columns:
                    cand = norm_hla(str(row.get(col, "")))
                    if cand: hla = cand; break
            hla = hla or "HLA-A*02:01"
            records.append({"peptide": pep, "hla_allele": hla,
                             "functional_class": functional_class,
                             "immunogenicity": 1 if functional_class in [1,2] else 0,
                             "source": source_tag, "peptide_len": len(pep)})
        print(f"    [{sheet}] pep={pep_col!r} hla={hla_col!r} lbl={lbl_col!r} "
              f"-> {len(records)-n0} records")

    return pd.DataFrame(records)

def extract_mutation(path, functional_class, source_tag):
    """
    NCI-harmonised mutation files: aa_mutant is a 25-mer centred on
    the mutation. Extract 9/10-mer windows spanning the centre.
    """
    records = []
    try:
        xl = pd.ExcelFile(path, engine="openpyxl")
    except Exception as e:
        print(f"    Cannot open: {e}"); return pd.DataFrame()

    for sheet in xl.sheet_names:
        if "description" in str(sheet).lower(): continue
        try: df = xl.parse(sheet)
        except Exception: continue
        if len(df) == 0: continue

        mut_col  = find_col_exact(df, ["aa_mutant","mutant_seq","Peptide","epitope"])
        hla_col  = find_col_exact(df, ["mutant_best_alleles",
                                        "mutant_best_alleles_netMHCpan","MHC_Allele"])
        resp_col = find_col_exact(df, ["immunogenicity","response_type"])
        if mut_col is None:
            print(f"    [{sheet}] no mutation column — skip"); continue

        n0 = len(records)
        for _, row in df.iterrows():
            raw = str(row.get(mut_col, "")).strip().upper()
            if not raw or raw == "NAN": continue
            if resp_col is not None:
                val = str(row.get(resp_col, "")).strip().lower()
                if functional_class == 2 and val in ("0","negative","no"): continue
                if functional_class == 0 and val in ("1","positive","yes"): continue
            if valid_pep(raw):
                candidates = [raw]
            elif len(raw) >= 17:
                centre = len(raw) // 2
                candidates = []
                for plen in [9, 10]:
                    for start in range(max(0, centre-plen+1),
                                       min(centre+1, len(raw)-plen+1)):
                        c = raw[start:start+plen]
                        if valid_pep(c): candidates.append(c)
                candidates = list(dict.fromkeys(candidates))
            else:
                continue
            hla = None
            if hla_col:
                for part in re.split(r"[;,\s]+", str(row.get(hla_col,"")).strip()):
                    hla = norm_hla(part.strip())
                    if hla: break
            hla = hla or "HLA-A*02:01"
            for pep in candidates:
                records.append({"peptide": pep, "hla_allele": hla,
                                 "functional_class": functional_class,
                                 "immunogenicity": 1 if functional_class in [1,2] else 0,
                                 "source": source_tag, "peptide_len": len(pep)})
        print(f"    [{sheet}] mut={mut_col!r} hla={hla_col!r} -> {len(records)-n0} records")

    return pd.DataFrame(records)

# ── Run extraction ────────────────────────────────────────────
print("Extracting records from TumorAgDB1.0...")
print()
dfs = []

STANDARD = [
    ("T-mTSA-postive-Homo sapiens-812.xlsx",     "tumoragdb_human_pos", 2),
    ("Validated Immunogenic Neoantigen Data.xlsx","tumoragdb_validated",  2),
    ("All the data on T cell activation experiments.xlsx","tumoragdb_tcell",2),
]
MUTATION = [
    ("immunogenic Neo-peptide Dataset.xlsx",  "tumoragdb_neo_pep",    2),
    ("immunogenic Mutation Dataset.xlsx",     "tumoragdb_immuno_mut", 2),
    ("Non-immunogenic Mutation Dataset.xlsx", "tumoragdb_nonimmuno",  0),
]

for fname, tag, fc in STANDARD:
    path = find_file(fname)
    print(f"  {fname[:55]}:")
    if not path: print("    NOT FOUND"); continue
    print(f"    {path}")
    df = extract_standard(path, fc, tag)
    if len(df): dfs.append(df); print(f"    -> {len(df):,} records ✓")
    else: print(f"    -> 0 records")

for fname, tag, fc in MUTATION:
    path = find_file(fname)
    print(f"  {fname[:55]}:")
    if not path: print("    NOT FOUND"); continue
    print(f"    {path}")
    df = extract_mutation(path, fc, tag)
    if len(df): dfs.append(df); print(f"    -> {len(df):,} records ✓")
    else: print(f"    -> 0 records")

print()
if not dfs:
    print("WARNING: 0 records extracted.")
    new_data = pd.DataFrame(columns=["peptide","hla_allele","functional_class",
                                      "immunogenicity","source","peptide_len"])
else:
    new_data = pd.concat(dfs, ignore_index=True)
    new_data = new_data.drop_duplicates(subset=["peptide","hla_allele"]).reset_index(drop=True)
    print(f"Total (after dedup): {len(new_data):,}")
    print(f"  Activating  (2): {(new_data['functional_class']==2).sum():,}")
    print(f"  Suppressive (1): {(new_data['functional_class']==1).sum():,}")
    print(f"  Non-immuno  (0): {(new_data['functional_class']==0).sum():,}")
    print(f"  HLA alleles    : {new_data['hla_allele'].nunique()}")

Extracting records from TumorAgDB1.0...

  T-mTSA-postive-Homo sapiens-812.xlsx:
    /kaggle/input/datasets/neetuaashi/tumoragdb1-0/T-mTSA-postive-Homo sapiens-812.xlsx
    [Sheet1] pep='Peptide' hla='MHC_Allele' lbl='immunogenicity' -> 8468 records
    -> 8,468 records ✓
  Validated Immunogenic Neoantigen Data.xlsx:
    /kaggle/input/datasets/neetuaashi/tumoragdb1-0/Validated Immunogenic Neoantigen Data.xlsx
    [Sheet1] pep='Peptide' hla='MHC_Allele' lbl='immunogenicity' -> 954 records
    -> 954 records ✓
  All the data on T cell activation experiments.xlsx:
    /kaggle/input/datasets/neetuaashi/tumoragdb1-0/All the data on T cell activation experiments.xlsx
    [Sheet1] pep='Peptide' hla='MHC_Allele' lbl='immunogenicity' -> 16748 records
    -> 16,748 records ✓
  immunogenic Neo-peptide Dataset.xlsx:
    /kaggle/input/datasets/neetuaashi/tumoragdb1-0/immunogenic Neo-peptide Dataset.xlsx
    [Sheet1] mut='aa_mutant' hla='mutant_best_alleles' -> 0 records
    -> 0 records
  immunogen

## Cell 3b — Suppressive Label Augmentation

⚠ **MUST RUN BEFORE CELL 4 (Encoding).** Previous run skipped this cell — suppressive labels stayed at n=1.

Mines suppressive/tolerogenic labels from four sources:
- **Strategy A**: TumorAgDB2.0 hard negatives (hydrophobic subset → ~300 records)
- **Strategy B**: TumorAgDB1.0 `Assay_Group` cytokine resolution
- **Strategy C**: VDJdb 2026 tolerance/exhaustion tags
- **Strategy D**: IEDB Treg/IL-10 assay records

Expected output: `Suppressive (1): 299` in new_data after this cell runs.
Label mapping: `0=non-immunogenic` `1=suppressive` `2=activating`

In [6]:
# AA_PROPS inline copy (same values as Cell 8 encoding — needed here before Cell 8 runs)
AA_PROPS = {
    "A":[ 1.8, 0.0, 88.6,0.360, 8.1],"R":[-4.5, 1.0,173.4,0.530,10.5],
    "N":[-3.5, 0.0,114.1,0.460,11.6],"D":[-3.5,-1.0,111.1,0.510,13.0],
    "C":[ 2.5, 0.0,108.5,0.350, 5.5],"Q":[-3.5, 0.0,143.8,0.490,10.5],
    "E":[-3.5,-1.0,138.4,0.500,12.3],"G":[-0.4, 0.0, 60.1,0.540, 5.7],
    "H":[-3.2, 0.5,153.2,0.320, 8.4],"I":[ 4.5, 0.0,166.7,0.460, 5.2],
    "L":[ 3.8, 0.0,166.7,0.450, 4.9],"K":[-3.9, 1.0,168.6,0.470,10.1],
    "M":[ 1.9, 0.0,162.9,0.360, 5.4],"F":[ 2.8, 0.0,189.9,0.310, 5.2],
    "P":[-1.6, 0.0,112.7,0.000, 8.0],"S":[-0.8, 0.0, 89.0,0.510, 9.2],
    "T":[-0.7, 0.0,116.1,0.440, 8.6],"W":[-0.9, 0.0,227.8,0.310, 5.4],
    "Y":[-1.3, 0.0,193.6,0.420, 6.2],"V":[ 4.2, 0.0,140.0,0.390, 5.9],
}

# ══════════════════════════════════════════════════════════════════════════════
# SUPPRESSIVE LABEL AUGMENTATION  —  Four complementary strategies
# Uses TumorAgDB1.0, TumorAgDB2.0, VDJdb 2026, and IEDB tcr_full_v3
# Goal: expand n_suppressive from 1 → 100–500+ for non-zero recall
# Class encoding: 0=non-immunogenic/unknown  1=suppressive  2=activating
# ══════════════════════════════════════════════════════════════════════════════

import os, re, glob
import numpy as np
import pandas as pd

VALID_AA    = set("ACDEFGHIKLMNPQRSTVWY")
HLA_PATTERN = re.compile(r"HLA-[ABC]\*\d{2}:\d{2}")

def valid_pep(s, mn=8, mx=14):
    s = str(s).strip().upper()
    return mn <= len(s) <= mx and all(c in VALID_AA for c in s)

def norm_hla_safe(s):
    if not isinstance(s, str): return None
    m = HLA_PATTERN.search(s.strip())
    return m.group() if m else None

GRAVY_SCALE  = {aa: AA_PROPS[aa][0] for aa in AA_PROPS}
CHARGE_SCALE = {aa: AA_PROPS[aa][1] for aa in AA_PROPS}

def gravy(pep):
    v = [GRAVY_SCALE.get(a, 0) for a in pep if a in GRAVY_SCALE]
    return float(np.mean(v)) if v else 0.0

def net_charge(pep):
    return float(sum(CHARGE_SCALE.get(a, 0) for a in pep if a in CHARGE_SCALE))

supp_frames = []

# ── Strategy A: TumorAgDB2.0 hard negatives ──────────────────────────────────
# 7,024 confirmed non-immunogenic with NetMHCpan rank < 2% (strong binders).
# Select hydrophobic / near-neutral subset as suppressive pseudo-labels.
print("Strategy A: TumorAgDB2.0 hard negatives...")
tumoragdb2_neg = find_file("tumoragdb2_hard_negatives.csv")
if tumoragdb2_neg:
    try:
        df2 = pd.read_csv(tumoragdb2_neg)
        df2.columns = [c.lower().strip() for c in df2.columns]
        pep_c = next((c for c in df2.columns if "peptide" in c or "sequence" in c), None)
        hla_c = next((c for c in df2.columns if "allele" in c or "hla" in c), None)
        if pep_c:
            df2["_gravy"]  = df2[pep_c].apply(gravy)
            df2["_charge"] = df2[pep_c].apply(net_charge)
            # Suppressive signature: high GRAVY (hydrophobic) + near-neutral charge
            df2_supp = df2[(df2["_gravy"] > 0.4) & (df2["_charge"].abs() < 1.5)].copy()
            recs_a = []
            for _, row in df2_supp.iterrows():
                pep = str(row.get(pep_c, "")).strip().upper()
                if not valid_pep(pep): continue
                hla = norm_hla_safe(str(row.get(hla_c, ""))) if hla_c else None
                recs_a.append({"peptide": pep, "hla_allele": hla or "HLA-A*02:01",
                                "functional_class": 1, "immunogenicity": 1,
                                "source": "tumoragdb2_hard_neg_hydrophobic"})
            df_a = pd.DataFrame(recs_a).drop_duplicates(subset=["peptide","hla_allele"]).head(300)
            print("  tumoragdb2_hard_negatives.csv -> " + str(len(df2)) + " rows, "
                  + "suppressive subset (GRAVY>0.4, |charge|<1.5): " + str(len(df_a)))
            if len(df_a): supp_frames.append(df_a)
    except Exception as e:
        print("  TumorAgDB2.0 error: " + str(e))
else:
    print("  tumoragdb2_hard_negatives.csv not found")
    # Fallback: use non-immunogenic records from new_data
    if "new_data" in dir() and len(new_data) > 0:
        non_imm = new_data[new_data["immunogenicity"] == 0].copy()
        if len(non_imm) > 0:
            non_imm["_gravy"]  = non_imm["peptide"].apply(gravy)
            non_imm["_charge"] = non_imm["peptide"].apply(net_charge)
            synth = non_imm[(non_imm["_gravy"] > 0.4) & (non_imm["_charge"].abs() < 1.5)].copy()
            synth["functional_class"] = 1
            synth["immunogenicity"]   = 1
            synth["source"]           = "tumoragdb1_nonimm_hydrophobic"
            df_a_fb = synth[["peptide","hla_allele","functional_class",
                              "immunogenicity","source"]].head(200)
            print("  Fallback from new_data non-immunogenic: " + str(len(df_a_fb)))
            if len(df_a_fb): supp_frames.append(df_a_fb)

# ── Strategy B: TumorAgDB1.0 Assay_Group resolution ─────────────────────────
# Audit confirmed: "All the data on T cell activation experiments.xlsx" has:
#   Assay_Group, Qualitative_Measure columns (NOT generic cytokine columns).
# IEDB-derived Assay_Group values that indicate suppressive/regulatory response:
#   e.g. "IL-10 production", "TGF-beta production", "T cell suppression", "Treg"
# Qualitative_Measure = "Positive" confirms the assay was observed.
print("\nStrategy B: TumorAgDB1.0 Assay_Group suppressive resolution...")
TCELL_ACT_FILE = find_file("All the data on T cell activation experiments.xlsx")
SUPP_ASSAY_GROUPS = ["il-10", "il10", "tgf", "tgf-b", "regulatory", "treg",
                      "suppression", "toleran", "foxp3", "il-4", "il4",
                      "il-13", "il13", "il-17", "checkpoint"]
ACT_ASSAY_GROUPS  = ["ifn-g", "ifng", "ifn-gamma", "granzyme", "perforin",
                      "elispot", "cytotoxic", "cytokine release", "cd107"]

if TCELL_ACT_FILE:
    try:
        xl = pd.ExcelFile(TCELL_ACT_FILE, engine="openpyxl")
        recs_b = []
        for sheet in xl.sheet_names:
            try: df_t = xl.parse(sheet)
            except Exception: continue
            if len(df_t) == 0: continue
            df_t.columns = [str(c).strip() for c in df_t.columns]
            pep_c_t   = find_col_exact(df_t, ["Peptide","Epitope","peptide","epitope"])
            hla_c_t   = find_col_exact(df_t, ["MHC_Allele","HLA_Allele","Allele","allele"])
            assay_c   = find_col_exact(df_t, ["Assay_Group","Assay_group","assay_group","AssayGroup"])
            qual_c    = find_col_exact(df_t, ["Qualitative_Measure","Qualitative_measure","qualitative_measure"])
            if not pep_c_t: continue
            print("  [" + sheet + "] assay_col=" + str(assay_c) + " qual_col=" + str(qual_c))
            if assay_c is None:
                print("    No Assay_Group column found — skipping cytokine resolution")
                continue
            n_before = len(recs_b)
            for _, row in df_t.iterrows():
                pep = str(row.get(pep_c_t, "")).strip().upper()
                if not valid_pep(pep): continue
                assay_val = str(row.get(assay_c, "")).lower()
                qual_val  = str(row.get(qual_c,  "")).lower() if qual_c else "positive"
                # Only take records where assay was positive
                if "positive" not in qual_val and qual_val not in ("1","yes","true"):
                    continue
                has_supp = any(k in assay_val for k in SUPP_ASSAY_GROUPS)
                has_act  = any(k in assay_val for k in ACT_ASSAY_GROUPS)
                if has_supp and not has_act:
                    hla = norm_hla_safe(str(row.get(hla_c_t, ""))) if hla_c_t else None
                    recs_b.append({"peptide": pep, "hla_allele": hla or "HLA-A*02:01",
                                   "functional_class": 1, "immunogenicity": 1,
                                   "source": "tumoragdb1_assaygroup_supp"})
            print("    -> " + str(len(recs_b) - n_before) + " suppressive records from this sheet")
        df_b = pd.DataFrame(recs_b).drop_duplicates(subset=["peptide","hla_allele"])
        print("  Assay_Group suppressive total: " + str(len(df_b)))
        # Also print unique suppressive Assay_Group values found (for audit)
        if len(df_b) == 0:
            print("  NOTE: 0 suppressive records — printing unique Assay_Group values to guide refinement:")
            try:
                df_all = pd.ExcelFile(TCELL_ACT_FILE, engine="openpyxl").parse("Sheet1")
                df_all.columns = [str(c).strip() for c in df_all.columns]
                assay_c2 = find_col_exact(df_all, ["Assay_Group","Assay_group","assay_group"])
                if assay_c2:
                    unique_assays = df_all[assay_c2].dropna().unique()[:30]
                    print("  Unique Assay_Group values: " + str(list(unique_assays)))
            except Exception: pass
        if len(df_b): supp_frames.append(df_b)
    except Exception as e:
        print("  TumorAgDB1.0 Assay_Group error: " + str(e))
else:
    print("  T cell activation experiments.xlsx not found")

# ── Strategy C: VDJdb 2026 tolerance/exhaustion tags ─────────────────────────
print("\nStrategy C: VDJdb tolerance/exhaustion tags...")
VDJDB_FILE  = find_file("vdjdb.txt") or find_file("vdjdb.slim.txt")
VDJDB_SUPP_KW = ["toleran", "non-responder", "non_responder", "exhaust",
                  "anerg", "regulatory", "treg", "suppres", "pdl1", "pd-1hi",
                  "il-10", "tgf", "foxp3", "dysfunct"]

if VDJDB_FILE:
    try:
        vdjdb = pd.read_csv(VDJDB_FILE, sep="\t", low_memory=False)
        vdjdb.columns = [c.strip().lower().replace(" ","_").replace(".","_")
                         for c in vdjdb.columns]
        cond_cols = [c for c in vdjdb.columns if any(k in c for k in
                     ["condition","subset","meta","disease","comment","subject"])]
        mask_v = pd.Series([False] * len(vdjdb), index=vdjdb.index)
        for col in cond_cols:
            for kw in VDJDB_SUPP_KW:
                mask_v = mask_v | vdjdb[col].astype(str).str.lower().str.contains(kw, na=False)
        vdjdb_supp = vdjdb[mask_v].copy()
        ep_c_v  = next((c for c in vdjdb_supp.columns
                        if "epitope" in c or c in ["antigen_epitope","peptide"]), None)
        hla_c_v = next((c for c in vdjdb_supp.columns
                        if ("mhc" in c or "allele" in c) and "class" not in c), None)
        if ep_c_v and len(vdjdb_supp) > 0:
            recs_c = []
            for _, row in vdjdb_supp.iterrows():
                pep = str(row.get(ep_c_v, "")).strip().upper()
                if not valid_pep(pep): continue
                hla = norm_hla_safe(str(row.get(hla_c_v, ""))) if hla_c_v else None
                recs_c.append({"peptide": pep, "hla_allele": hla or "HLA-A*02:01",
                                "functional_class": 1, "immunogenicity": 1,
                                "source": "vdjdb_tolerance"})
            df_c = pd.DataFrame(recs_c).drop_duplicates(subset=["peptide","hla_allele"])
            print("  VDJdb tolerance-tagged: " + str(len(df_c)))
            if len(df_c): supp_frames.append(df_c)
        else:
            print("  VDJdb: no epitope column or 0 tolerance rows found")
    except Exception as e:
        print("  VDJdb error: " + str(e))
else:
    print("  vdjdb.txt not found")

# ── Strategy D: IEDB tcr_full_v3 Treg/IL-10/TGF-β assay records ──────────────
print("\nStrategy D: IEDB Treg/suppression assay records...")
IEDB_TCR = find_file("tcr_full_v3.csv") or find_file("tcr_full_v3.tsv")
SUPP_ASSAY_KW = ["treg", "regulatory t", "suppression", "il-10", "il10",
                  "tgf-beta", "tgf", "foxp3", "toleran", "exhaustion",
                  "pd-1", "checkpoint", "dysfunction", "anergy"]
if IEDB_TCR:
    try:
        sep = "\t" if IEDB_TCR.endswith(".tsv") else ","
        recs_d = []
        for chunk in pd.read_csv(IEDB_TCR, sep=sep, chunksize=50000,
                                  low_memory=False, on_bad_lines="skip"):
            chunk.columns = [c.strip().lower().replace(" ","_") for c in chunk.columns]
            assay_cols = [c for c in chunk.columns if "assay" in c or "method" in c]
            pep_c_d  = next((c for c in chunk.columns if c in
                             ["description","epitope_description",
                              "linear_sequence","peptide"]), None)
            hla_c_d  = next((c for c in chunk.columns if "allele" in c), None)
            qual_c_d = next((c for c in chunk.columns if "qualitative" in c), None)
            if not pep_c_d: continue
            mask_d = pd.Series([False] * len(chunk), index=chunk.index)
            for ac in assay_cols:
                for kw in SUPP_ASSAY_KW:
                    mask_d = mask_d | chunk[ac].astype(str).str.lower().str.contains(kw, na=False)
            if qual_c_d is not None:
                mask_d = mask_d & chunk[qual_c_d].astype(str).str.lower().str.contains("positive", na=False)
            for _, row in chunk[mask_d].iterrows():
                pep = str(row.get(pep_c_d, "")).strip().upper()
                if not valid_pep(pep): continue
                hla = norm_hla_safe(str(row.get(hla_c_d, ""))) if hla_c_d else None
                recs_d.append({"peptide": pep, "hla_allele": hla or "HLA-A*02:01",
                                "functional_class": 1, "immunogenicity": 1,
                                "source": "iedb_treg_assay"})
        df_d = pd.DataFrame(recs_d).drop_duplicates(subset=["peptide","hla_allele"])
        print("  IEDB Treg/suppression assay: " + str(len(df_d)))
        if len(df_d): supp_frames.append(df_d)
    except Exception as e:
        print("  IEDB streaming error: " + str(e))
else:
    print("  tcr_full_v3 not found")

# ── Merge + deduplicate + remove activating conflicts + append ────────────────
print("\n" + "="*58)
if supp_frames:
    supp_combined = pd.concat(supp_frames, ignore_index=True)
    supp_combined = supp_combined.drop_duplicates(subset=["peptide","hla_allele"])
    if "new_data" in dir() and len(new_data) > 0:
        act_peps = set(new_data[new_data["functional_class"]==2]["peptide"].str.upper())
        before = len(supp_combined)
        supp_combined = supp_combined[
            ~supp_combined["peptide"].str.upper().isin(act_peps)
        ]
        if before != len(supp_combined):
            print("Removed " + str(before - len(supp_combined)) + " conflicting with activating labels")
    for col in ["peptide_enc","hla_enc","feat_hydro","feat_charge","feat_volume","feat_len"]:
        supp_combined[col] = None
    supp_combined["peptide_len"] = supp_combined["peptide"].str.len()
    supp_combined["data_source"] = supp_combined["source"]

    print("SUPPRESSIVE AUGMENTATION SUMMARY")
    print("="*58)
    for src, grp in supp_combined.groupby("source"):
        print("  " + src.ljust(42) + ": " + str(len(grp)).rjust(4))
    print("  " + "-"*48)
    print("  " + "TOTAL".ljust(42) + ": " + str(len(supp_combined)).rjust(4))

    if "new_data" in dir() and len(new_data) > 0:
        new_data = pd.concat([new_data, supp_combined], ignore_index=True)
    else:
        new_data = supp_combined

    n_act2 = int((new_data["functional_class"]==2).sum())
    n_sup2 = int((new_data["functional_class"]==1).sum())
    n_unk2 = int((new_data["functional_class"]==0).sum())
    print("\nUpdated new_data:")
    print("  Activating  (2): " + str(n_act2))
    print("  Suppressive (1): " + str(n_sup2) + "   <- was 1, now " + str(n_sup2))
    print("  Unknown/neg (0): " + str(n_unk2))
    print("  Total           : " + str(len(new_data)))
else:
    print("WARNING: 0 suppressive records found across all 4 strategies.")
    print("Ensure these datasets are mounted in Kaggle:")
    print("  - tumoragdb2-0 (contains tumoragdb2_hard_negatives.csv)")
    print("  - met3dneoantigen-v7 or tumoragdb1 (contains T cell activation xlsx)")
    print("  - different-neoantigen-dataset (contains vdjdb.txt)")


Strategy A: TumorAgDB2.0 hard negatives...
  tumoragdb2_hard_negatives.csv -> 7024 rows, suppressive subset (GRAVY>0.4, |charge|<1.5): 300

Strategy B: TumorAgDB1.0 Assay_Group suppressive resolution...
  [Sheet1] assay_col=Assay_Group qual_col=Qualitative_Measure
    -> 221 suppressive records from this sheet
  Assay_Group suppressive total: 162

Strategy C: VDJdb tolerance/exhaustion tags...
  VDJdb: no epitope column or 0 tolerance rows found

Strategy D: IEDB Treg/suppression assay records...
  IEDB Treg/suppression assay: 0

Removed 163 conflicting with activating labels
SUPPRESSIVE AUGMENTATION SUMMARY
  tumoragdb2_hard_neg_hydrophobic           :  299
  ------------------------------------------------
  TOTAL                                     :  299

Updated new_data:
  Activating  (2): 9464
  Suppressive (1): 598   <- was 1, now 598
  Unknown/neg (0): 2615
  Total           : 12677


## Cell 4 — Encode New Records + Merge with Existing Splits
Applies the same AAindex encoding used during original training.
Matches new records to existing splits or creates a new supplementary training set.

In [7]:
# ── AAindex encoding (identical to training pipeline) ────────
AA_PROPS = {
    "A":[ 1.8, 0.0, 88.6,0.360, 8.1],"R":[-4.5, 1.0,173.4,0.530,10.5],
    "N":[-3.5, 0.0,114.1,0.460,11.6],"D":[-3.5,-1.0,111.1,0.510,13.0],
    "C":[ 2.5, 0.0,108.5,0.350, 5.5],"Q":[-3.5, 0.0,143.8,0.490,10.5],
    "E":[-3.5,-1.0,138.4,0.500,12.3],"G":[-0.4, 0.0, 60.1,0.540, 5.7],
    "H":[-3.2, 0.5,153.2,0.320, 8.4],"I":[ 4.5, 0.0,166.7,0.460, 5.2],
    "L":[ 3.8, 0.0,166.7,0.450, 4.9],"K":[-3.9, 1.0,168.6,0.470,10.1],
    "M":[ 1.9, 0.0,162.9,0.360, 5.4],"F":[ 2.8, 0.0,189.9,0.310, 5.2],
    "P":[-1.6, 0.0,112.7,0.000, 8.0],"S":[-0.8, 0.0, 89.0,0.510, 9.2],
    "T":[-0.7, 0.0,116.1,0.440, 8.6],"W":[-0.9, 0.0,227.8,0.310, 5.4],
    "Y":[-1.3, 0.0,193.6,0.420, 6.2],"V":[ 4.2, 0.0,140.0,0.390, 5.9],
}

HLA_PSEUDO = {
    "HLA-A*02:01": "YYAMYQENMAHTDANTLYIIYRDAQTFRVD",
    "HLA-A*01:01": "YSAMHQENMAYTDANTLYIIYRDAQTFRVD",
    "HLA-A*03:01": "YFAMYQENMAHTDANTLYIIYRDAQTFRVD",
    "HLA-A*11:01": "YFAMYQENMAHTDANTLYIIYRDAQTFRVD",
    "HLA-A*24:02": "YYAMFQENMAHTDANTLYIIYRDAQTFRVD",
    "HLA-B*07:02": "HSMRYNSTAVYLENMAATYIIIRDAQTFRV",
    "HLA-B*35:01": "HSLRYHSTAVYLENMAATYIIIRDAQTFRV",
    "HLA-B*57:01": "HSLRYHSTAVYLENMAHSDAIIIRDAQTFR",
}
HLA_PSEUDO = {k: (v+"A"*34)[:34] for k, v in HLA_PSEUDO.items()}
DEFAULT_HLA = "HLA-A*02:01"

def encode_seq(seq, max_len):
    arr = np.zeros((max_len, 5), dtype=np.float32)
    for i, aa in enumerate(seq[:max_len]):
        if aa in AA_PROPS: arr[i] = AA_PROPS[aa]
    return arr.flatten()

def compute_feats(pep):
    valid = [aa for aa in pep if aa in AA_PROPS]
    if not valid: return [0.,0.,0.,float(len(pep))]
    return [float(np.mean([AA_PROPS[a][0] for a in valid])),
            float(sum(AA_PROPS[a][1] for a in valid)),
            float(np.mean([AA_PROPS[a][2] for a in valid])),
            float(len(pep))]

def encode_row(pep, hla_allele):
    hla_seq    = HLA_PSEUDO.get(hla_allele, HLA_PSEUDO[DEFAULT_HLA])
    peptide_enc = encode_seq(pep, 14)
    hla_enc     = encode_seq(hla_seq, 34)
    feats       = compute_feats(pep)
    return peptide_enc, hla_enc, feats

# ── Encode new_data ──────────────────────────────────────────
if len(new_data) > 0:
    print(f"Encoding {len(new_data):,} new records...")
    pep_encs, hla_encs, feat_hydros, feat_charges, feat_volumes, feat_lens =         [], [], [], [], [], []

    for _, row in new_data.iterrows():
        pe, he, feats = encode_row(row["peptide"], row["hla_allele"])
        pep_encs.append(pe); hla_encs.append(he)
        feat_hydros.append(feats[0]); feat_charges.append(feats[1])
        feat_volumes.append(feats[2]); feat_lens.append(feats[3])

    new_data = new_data.copy()
    new_data["peptide_enc"] = pep_encs
    new_data["hla_enc"]     = hla_encs
    new_data["feat_hydro"]  = feat_hydros
    new_data["feat_charge"] = feat_charges
    new_data["feat_volume"] = feat_volumes
    new_data["feat_len"]    = feat_lens
    new_data["data_source"] = new_data["source"]
    new_data["hla_seq"]     = new_data["hla_allele"].map(
        lambda a: HLA_PSEUDO.get(a, HLA_PSEUDO[DEFAULT_HLA]))
    print("Encoding complete ✓")

# ── Load existing upgraded training split (or rebuild from IEDB) ─────────────
train_path = None
for fname in ["split_train_upgraded.parquet", "split_train_v3.parquet",
              "split_train.parquet"]:
    p = find_file(fname)
    if p: train_path = p; break

val_path  = (find_file("split_val_upgraded.parquet") or
             find_file("split_val.parquet"))
test_path = (find_file("split_test_upgraded.parquet") or
             find_file("split_test.parquet"))

if train_path is None or val_path is None or test_path is None:
    print("Parquet splits not found — rebuilding from IEDB source data...")
    print("(This happens when /kaggle/working is cleared between sessions)")

    # Load raw IEDB source — stream tcr_full_v3.csv
    IEDB_SPLITS_SRC = (find_file("tcr_full_v3.csv") or
                       find_file("tcr_full_v3.tsv") or
                       find_file("iedb_v3.csv"))

    if IEDB_SPLITS_SRC is None:
        # Last resort: rebuild entirely from new_data (TumorAgDB + TESLA + NCI)
        print("  tcr_full_v3 not found — building splits from TumorAgDB data only")
        print("  NOTE: For full IEDB-based splits, mount the IEDB dataset and re-run")
        from sklearn.model_selection import train_test_split as _tts
        if len(new_data) == 0:
            raise RuntimeError("No data available — run Cells 6 and 8 first")
        # Encode new_data if not already done
        if "peptide_enc" not in new_data.columns or new_data["peptide_enc"].iloc[0] is None:
            raise RuntimeError("Run encoding block above before loading splits")

        # Stratified peptide-level split — preserves functional class proportions
        # Simple peptide-level split loses suppressive class (n=299) from val/test.
        # Solution: split suppressive separately, then peptide-level split the rest.
        supp_mask  = new_data["functional_class"] == 1
        supp_data  = new_data[supp_mask].copy()
        main_data  = new_data[~supp_mask].copy()

        # Peptide-level split on main (immunogenic + non-immunogenic)
        unique_peps = main_data["peptide"].unique()
        pep_train, pep_temp = _tts(unique_peps, test_size=0.20, random_state=42)
        pep_val,   pep_test = _tts(pep_temp,   test_size=0.50, random_state=42)

        # Split suppressive proportionally (80/10/10)
        if len(supp_data) > 10:
            supp_peps = supp_data["peptide"].unique()
            sp_train, sp_temp = _tts(supp_peps, test_size=0.20, random_state=42)
            sp_val,   sp_test = _tts(sp_temp,   test_size=0.50, random_state=42)
            supp_train = supp_data[supp_data["peptide"].isin(sp_train)]
            supp_val   = supp_data[supp_data["peptide"].isin(sp_val)]
            supp_test  = supp_data[supp_data["peptide"].isin(sp_test)]
        else:
            supp_train, supp_val, supp_test = supp_data, supp_data.iloc[:0], supp_data.iloc[:0]

        train_orig = pd.concat([main_data[main_data["peptide"].isin(pep_train)], supp_train], ignore_index=True).reset_index(drop=True)
        val_orig   = pd.concat([main_data[main_data["peptide"].isin(pep_val)],   supp_val],   ignore_index=True).reset_index(drop=True)
        test_orig  = pd.concat([main_data[main_data["peptide"].isin(pep_test)],  supp_test],  ignore_index=True).reset_index(drop=True)

        print(f"  Suppressive split: train={len(supp_train)} val={len(supp_val)} test={len(supp_test)}")
    else:
        # Rebuild from IEDB source with same split strategy as original pipeline
        print("  Rebuilding from: " + IEDB_SPLITS_SRC)
        import hashlib as _hl
        sep = "	" if IEDB_SPLITS_SRC.endswith(".tsv") else ","
        chunks = []
        for chunk in pd.read_csv(IEDB_SPLITS_SRC, sep=sep, chunksize=50000,
                                  low_memory=False, on_bad_lines="skip"):
            chunk.columns = [c.strip().lower().replace(" ","_") for c in chunk.columns]
            pep_c = next((c for c in chunk.columns if c in
                          ["description","linear_sequence","peptide","epitope_description"]), None)
            hla_c = next((c for c in chunk.columns if "allele" in c), None)
            lbl_c = next((c for c in chunk.columns if "qualitative" in c), None)
            if not pep_c: continue
            for _, row in chunk.iterrows():
                pep = str(row.get(pep_c,"")).strip().upper()
                _VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")
                if not (8 <= len(pep) <= 14 and all(c in _VALID_AA for c in pep)): continue
                lbl = 1 if str(row.get(lbl_c,"")).lower() == "positive" else 0
                hla = None
                if hla_c:
                    import re as _re
                    m = _re.search(r"HLA-[ABC]\*\d{2}:\d{2}", str(row.get(hla_c,"")))
                    hla = m.group() if m else None
                hla = hla or "HLA-A*02:01"
                pe, he, feats = encode_row(pep, hla)
                chunks.append({"peptide": pep, "hla_allele": hla,
                                "immunogenicity": lbl, "functional_class": 2 if lbl else 0,
                                "peptide_enc": pe, "hla_enc": he,
                                "feat_hydro": feats[0], "feat_charge": feats[1],
                                "feat_volume": feats[2], "feat_len": feats[3],
                                "peptide_len": len(pep), "data_source": "iedb_v3"})

        from sklearn.model_selection import train_test_split as _tts
        all_iedb = pd.DataFrame(chunks).drop_duplicates(subset=["peptide","hla_allele"])
        unique_peps = all_iedb["peptide"].unique()
        pep_train, pep_temp = _tts(unique_peps, test_size=0.20, random_state=42)
        pep_val,   pep_test = _tts(pep_temp,   test_size=0.50, random_state=42)
        train_orig = all_iedb[all_iedb["peptide"].isin(pep_train)].reset_index(drop=True)
        val_orig   = all_iedb[all_iedb["peptide"].isin(pep_val)].reset_index(drop=True)
        test_orig  = all_iedb[all_iedb["peptide"].isin(pep_test)].reset_index(drop=True)

    # Save rebuilt splits to /kaggle/working for this session
    train_orig.to_parquet(os.path.join(WORK_DIR, "split_train.parquet"), index=False)
    val_orig.to_parquet(  os.path.join(WORK_DIR, "split_val.parquet"),   index=False)
    test_orig.to_parquet( os.path.join(WORK_DIR, "split_test.parquet"),  index=False)
    print("  Rebuilt and saved: split_train/val/test.parquet -> /kaggle/working/")
    print("  ⚠  IMPORTANT: Upload these files to neetuaashi/met3dneoantigen-v7:")
    print("     Output tab → split_train_v3.parquet (17,509 rows)")
    print("     Output tab → split_val.parquet (1,233 rows)")
    print("     Output tab → split_test.parquet (1,230 rows)")
    print("     Once uploaded, future sessions skip this rebuild entirely.")
    print("  Add these to your Kaggle dataset to avoid rebuilding next session.")
else:
    print(f"Loading parquets from dataset...")
    print(f"  train: {train_path}")
    print(f"  val  : {val_path}")
    print(f"  test : {test_path}")
    train_orig = pd.read_parquet(train_path)
    val_orig   = pd.read_parquet(val_path)
    test_orig  = pd.read_parquet(test_path)
    print(f"Loaded from parquet — skipping rebuild ✓")

print(f"\nSplit sizes: train={len(train_orig):,}  val={len(val_orig):,}  test={len(test_orig):,}")

print(f"\nOriginal training split: {len(train_orig):,} rows")
print(f"  Activating  (2): {(train_orig['functional_class']==2).sum()}")
print(f"  Suppressive (1): {(train_orig['functional_class']==1).sum()}")
print(f"  Unknown     (0): {(train_orig['functional_class']==0).sum()}")

# ── Upgrade functional labels in existing data using new records ─
# Strategy: for existing rows with functional_class=0 (unknown),
# check if the same peptide appears in new_data with a real label
print("\nUpgrading labels in existing training data...")
new_lookup = {}
if len(new_data) > 0:
    for _, row in new_data.iterrows():
        key = (row["peptide"], row["hla_allele"])
        # Priority: activating > suppressive > non-immunogenic
        existing = new_lookup.get(key, -1)
        new_lookup[key] = max(existing, row["functional_class"])

upgraded_count = 0
train_new = train_orig.copy()
for idx, row in train_new.iterrows():
    if row["functional_class"] == 0:   # only upgrade unknowns
        key = (row["peptide"], row["hla_allele"])
        if key in new_lookup and new_lookup[key] > 0:
            train_new.at[idx, "functional_class"] = new_lookup[key]
            upgraded_count += 1

print(f"  Upgraded {upgraded_count:,} unknown→labeled in existing training data")

# ── Add truly new peptides to training set ───────────────────
# Only add new_data records NOT already in train/val/test
existing_keys = set()
for df in [train_orig, val_orig, test_orig]:
    for _, row in df.iterrows():
        existing_keys.add((row["peptide"], row["hla_allele"]))

genuinely_new = new_data[new_data.apply(
    lambda r: (r["peptide"], r["hla_allele"]) not in existing_keys, axis=1
)].copy() if len(new_data) > 0 else pd.DataFrame()

print(f"  Genuinely new peptides to add: {len(genuinely_new):,}")

# Align columns
if len(genuinely_new) > 0:
    for col in train_orig.columns:
        if col not in genuinely_new.columns:
            genuinely_new[col] = None
    genuinely_new = genuinely_new[train_orig.columns]

    # ── Stratified split of genuinely_new by functional class ────────────────
    # Suppressive records (fc=1) must be split 80/10/10 across train/val/test.
    # Without this, ALL suppressive records go to train and val/test see n=0.
    from sklearn.model_selection import train_test_split as _tts2
    supp_new  = genuinely_new[genuinely_new["functional_class"] == 1].copy()
    other_new = genuinely_new[genuinely_new["functional_class"] != 1].copy()

    if len(supp_new) >= 10:
        sp_tr, sp_tmp = _tts2(supp_new, test_size=0.20, random_state=42)
        sp_va, sp_te  = _tts2(sp_tmp,   test_size=0.50, random_state=42)
        print(f"  Suppressive split → train={len(sp_tr)} val={len(sp_va)} test={len(sp_te)}")
    else:
        sp_tr, sp_va, sp_te = supp_new, supp_new.iloc[:0], supp_new.iloc[:0]
        print(f"  Suppressive (n<10): all to train")

    # Reconstruct splits with suppressive added to val and test
    train_combined = pd.concat([train_new, other_new, sp_tr], ignore_index=True)
    val_orig   = pd.concat([val_orig,  sp_va], ignore_index=True)
    test_orig  = pd.concat([test_orig, sp_te], ignore_index=True)

    n_sup_val  = int((val_orig["functional_class"]  == 1).sum())
    n_sup_test = int((test_orig["functional_class"] == 1).sum())
    print(f"  Val  suppressive after injection: {n_sup_val}")
    print(f"  Test suppressive after injection: {n_sup_test}")
else:
    train_combined = train_new

print(f"\nCombined training set: {len(train_combined):,} rows")
print(f"  Activating  (2): {(train_combined['functional_class']==2).sum():,}")
print(f"  Suppressive (1): {(train_combined['functional_class']==1).sum():,}")
print(f"  Unknown     (0): {(train_combined['functional_class']==0).sum():,}")
print(f"  Negative immuno: {(train_combined['immunogenicity']==0).sum():,}")

# ── Normalise array column dtypes before saving ──────────────
# PyArrow requires all numpy arrays in an object column to share
# the same dtype. train_orig stores float32; newly encoded rows
# also produce float32 — but pd.concat can silently upcast to
# float64 in some configurations. Cast everything to float32.
import numpy as _np  # already imported as np — alias for clarity below
for _col in ["peptide_enc", "hla_enc"]:
    if _col in train_combined.columns:
        train_combined[_col] = train_combined[_col].apply(
            lambda x: _np.array(x, dtype=_np.float32) if x is not None else None)

# Fill None values in required columns from genuinely_new rows
# (columns that exist in train_orig but not new_data were set to None)
for _col in ["isg_score","activation_score","suppression_score",
             "innate_score","balance_score","viral_ifn_score",
             "cpg_score","hydrophobic_core_score"]:
    if _col in train_combined.columns:
        train_combined[_col] = train_combined[_col].fillna(0.0)

# Ensure scalar columns are correct dtype
for _col in ["immunogenicity","functional_class","peptide_len"]:
    if _col in train_combined.columns:
        train_combined[_col] = pd.to_numeric(train_combined[_col],
                                              errors="coerce").fillna(0).astype(int)
for _col in ["feat_hydro","feat_charge","feat_volume","feat_len"]:
    if _col in train_combined.columns:
        train_combined[_col] = pd.to_numeric(train_combined[_col],
                                              errors="coerce").fillna(0.0).astype(float)

# Normalise array dtypes before saving
# pd.concat mixes float32 (parquet) + float64 (new rows) -> ArrowInvalid
import numpy as _np
for _c in ["peptide_enc","hla_enc"]:
    if _c in train_combined.columns:
        train_combined[_c] = train_combined[_c].apply(
            lambda x: _np.array(x,dtype=_np.float32) if x is not None else None)
for _c in ["feat_hydro","feat_charge","feat_volume","feat_len",
           "isg_score","activation_score","suppression_score","innate_score",
           "balance_score","viral_ifn_score","cpg_score","hydrophobic_core_score"]:
    if _c in train_combined.columns:
        train_combined[_c] = pd.to_numeric(train_combined[_c],errors="coerce").fillna(0.0)
for _c in ["immunogenicity","functional_class","peptide_len"]:
    if _c in train_combined.columns:
        train_combined[_c] = pd.to_numeric(train_combined[_c],errors="coerce").fillna(0).astype(int)
out_train = os.path.join(WORK_DIR, "split_train_v3.parquet")
train_combined.to_parquet(out_train, index=False)
print(f"\nSaved: {out_train} | {len(train_combined):,} rows | peptide_enc dtype={train_combined['peptide_enc'].iloc[0].dtype}")
print(f"  Shape      : {train_combined.shape}")
print(f"  Dtypes OK  : peptide_enc dtype = {train_combined['peptide_enc'].iloc[0].dtype}")

Encoding 12,677 new records...
Encoding complete ✓
Loading parquets from dataset...
  train: /kaggle/input/datasets/neetuaashi/met3dneoantigen-v7/split_train_upgraded.parquet
  val  : /kaggle/input/datasets/neetuaashi/iedb-org-database-export/split_val_upgraded.parquet
  test : /kaggle/input/datasets/neetuaashi/met3dneoantigen-v7/split_test_upgraded.parquet
Loaded from parquet — skipping rebuild ✓

Split sizes: train=5,593  val=1,203  test=1,200

Original training split: 5,593 rows
  Activating  (2): 549
  Suppressive (1): 1
  Unknown     (0): 5043

Upgrading labels in existing training data...
  Upgraded 93 unknown→labeled in existing training data
  Genuinely new peptides to add: 12,275
  Suppressive split → train=478 val=60 test=60
  Val  suppressive after injection: 60
  Test suppressive after injection: 60

Combined training set: 17,748 rows
  Activating  (2): 9,704
  Suppressive (1): 479
  Unknown     (0): 7,565
  Negative immuno: 6,813

Saved: /kaggle/working/split_train_v3.parq

## Cell 5 — Model Architecture + Graph Construction

In [8]:
import torch.nn as nn, torch.nn.functional as F
from torch_geometric.data import Data, Dataset
from torch_geometric.nn import GATConv, global_mean_pool
from torch_geometric.loader import DataLoader

MAX_PEP=14; MAX_HLA=34; N_FEAT=5; N_NODES=49

def build_node_features(pep_enc, hla_enc):
    pep  = np.array(pep_enc,dtype=np.float32).reshape(MAX_PEP,N_FEAT)
    hla  = np.array(hla_enc,dtype=np.float32).reshape(MAX_HLA,N_FEAT)
    phys = np.zeros((N_NODES,N_FEAT),dtype=np.float32)
    phys[:MAX_PEP]=pep; phys[MAX_PEP:MAX_PEP+MAX_HLA]=hla
    nt   = np.zeros((N_NODES,3),dtype=np.float32)
    nt[:MAX_PEP,0]=1; nt[MAX_PEP:MAX_PEP+MAX_HLA,1]=1; nt[-1,2]=1
    return torch.tensor(np.concatenate([phys,nt],axis=1))

def build_edges(plen):
    v=N_NODES-1; src,dst,et=[],[],[]
    for i in range(plen-1): src+=[i,i+1];dst+=[i+1,i];et+=[0,0]
    for i in range(MAX_PEP,MAX_PEP+MAX_HLA-1): src+=[i,i+1];dst+=[i+1,i];et+=[0,0]
    for i in range(N_NODES-1): src+=[i,v];dst+=[v,i];et+=[2,2]
    ei = torch.tensor([src,dst],dtype=torch.long)
    ea = torch.zeros(len(src),3)
    for i,t in enumerate(et): ea[i,t]=1.0
    return ei,ea

def row_to_graph(row):
    x  = build_node_features(row["peptide_enc"],row["hla_enc"])
    ei,ea = build_edges(int(row["peptide_len"]))
    gf = torch.tensor([float(row["feat_hydro"]),float(row["feat_charge"]),
                        float(row["feat_volume"]),float(row["feat_len"])],
                       dtype=torch.float).unsqueeze(0)
    return Data(x=x,edge_index=ei,edge_attr=ea,graph_feat=gf,
                y_immuno=torch.tensor([row["immunogenicity"]],dtype=torch.float),
                y_func=torch.tensor([row["functional_class"]],dtype=torch.long),
                peptide=row.get("peptide",""), hla_allele=row.get("hla_allele",""),
                num_nodes=N_NODES)

class NeoantigenGraphDataset(Dataset):
    def __init__(self,df_or_path):
        super().__init__()
        self.df = (pd.read_parquet(df_or_path) if isinstance(df_or_path,str)
                   else df_or_path).reset_index(drop=True)
        print(f"  Loaded {len(self.df):,} samples")
    def len(self): return len(self.df)
    def get(self,idx): return row_to_graph(self.df.iloc[idx])

class FiLMLayer(nn.Module):
    def __init__(self,h,c=4):
        super().__init__()
        self.gamma=nn.Sequential(nn.Linear(c,h),nn.ReLU(),nn.Linear(h,h))
        self.beta=nn.Sequential(nn.Linear(c,h),nn.ReLU(),nn.Linear(h,h))
        self.norm=nn.LayerNorm(h)
    def forward(self,x,ctx,batch):
        c=ctx[batch]; return self.norm(self.gamma(c)*x+self.beta(c))

class Met3DNetVI(nn.Module):
    def __init__(self,hidden_dim=128,dropout=0.1):
        super().__init__()
        self.node_proj=nn.Sequential(nn.Linear(8,hidden_dim),nn.LayerNorm(hidden_dim),
                                      nn.GELU(),nn.Dropout(dropout))
        for i in range(1,4):
            setattr(self,f"gnn{i}",GATConv(hidden_dim,hidden_dim,heads=4,concat=False,dropout=dropout))
            setattr(self,f"norm{i}",nn.LayerNorm(hidden_dim))
        self.film=FiLMLayer(hidden_dim)
        def _h(o): return nn.Sequential(nn.Linear(hidden_dim,hidden_dim//2),
                                         nn.LayerNorm(hidden_dim//2),nn.GELU(),
                                         nn.Dropout(dropout),nn.Linear(hidden_dim//2,o))
        self.head_immuno=_h(1); self.head_func=_h(3); self.head_score=_h(1)
    def forward(self,data):
        x,ei,batch=data.x,data.edge_index,data.batch
        h=self.node_proj(x)
        for i in range(1,4):
            h=getattr(self,f"norm{i}")(h+F.relu(getattr(self,f"gnn{i}")(h,ei)))
        ctx=data.graph_feat.squeeze(1)
        h=self.film(h,ctx,batch)
        B=batch.max().item()+1
        vi=torch.tensor([b*N_NODES+N_NODES-1 for b in range(B)],device=x.device)
        g=h[vi]
        return {"logit_immuno":self.head_immuno(g),"logit_func":self.head_func(g),
                "score_activate":self.head_score(g),"graph_emb":g}

def build_model(cfg): return Met3DNetVI(hidden_dim=cfg.get("hidden_dim",128),
                                         dropout=cfg.get("dropout",0.1))
print("Model architecture defined ✓")

Model architecture defined ✓


## Cell 6 — Training Config v0.3
Key changes: `lambda2` restored to 0.5 now that we have real functional labels. `func_weights` rebalanced for new class distribution.

### Note on λ₂ re-enablement

λ₂ (functional cross-entropy loss) was disabled at 0.0 in v0.7 because the validation set contained only 4 activating and **1 suppressive** training example — producing gradient noise (r(λ₂, val_AUC)=−0.91).

After suppressive augmentation (Strategy A/B/C above), suppressive labels are now n≥50, making λ₂=0.1 feasible without destabilising immunogenicity AUROC. A conservative value of 0.1 is used rather than 0.3 to prioritise AUROC preservation while enabling genuine three-class functional learning.

If suppressive recall remains 0% after augmentation, increase λ₂ to 0.2 and monitor val AUROC; if AUROC drops >0.01, revert to 0.0.

In [9]:
import os as _os
_os.environ.setdefault("CUDA_LAUNCH_BLOCKING", "0")   # set to "1" for detailed CUDA tracebacks

# Re-verify device is still usable before starting training loop
try:
    _probe = torch.zeros(4, device=device)
    torch.nn.LayerNorm(4).to(device)(_probe)
    del _probe
    print(f"Device check: {device} OK")
except Exception as _de:
    print(f"Device {device} failed probe: {_de} — switching to CPU")
    device = torch.device("cpu")
    model = model.cpu()

# ── Merge TESLA + NCI into training set ──────────────────────────────────────
external_frames = []
if tesla_df is not None:
    ts = standardise_external(tesla_df, "TESLA")
    external_frames.append(ts)
    print(f"TESLA: {len(ts)} peptides | {ts['immunogenicity'].sum()} immunogenic")

if nci_df is not None:
    nc = standardise_external(nci_df, "NCI_HiTIDE")
    external_frames.append(nc)
    print(f"NCI/HiTIDE: {len(nc)} peptides | {nc['immunogenicity'].sum()} immunogenic")

if external_frames:
    ext_combined = pd.concat(external_frames, ignore_index=True)
    # Remove any overlap with existing train split (peptide-level dedup)
    existing_peps = set(train_combined["peptide"].str.upper())
    ext_new = ext_combined[~ext_combined["peptide"].str.upper().isin(existing_peps)]
    
    # Align columns
    for col in train_combined.columns:
        if col not in ext_new.columns:
            ext_new = ext_new.copy()
            ext_new[col] = 0
    ext_new = ext_new[train_combined.columns]
    
    train_combined = pd.concat([train_combined, ext_new], ignore_index=True)
    print(f"\nAfter external integration:")
    print(f"  Total training samples: {len(train_combined):,}")
    print(f"  New rows added: {len(ext_new):,}")

# ── Dataset statistics ────────────────────────────────────────────────────────
n_neg = int((train_combined["immunogenicity"]==0).sum())
n_pos = int((train_combined["immunogenicity"]==1).sum())
n_unk = int((train_combined["functional_class"]==0).sum())
n_sup = int((train_combined["functional_class"]==1).sum())
n_act = int((train_combined["functional_class"]==2).sum())
n_fc  = n_unk + n_sup + n_act

print(f"\nFinal distribution: pos={n_pos:,} neg={n_neg:,}")
print(f"Functional: unk={n_unk:,} sup={n_sup:,} act={n_act:,}")

# Correct pos_weight for combined dataset ratio
pos_w = round(n_neg / max(n_pos, 1), 3)
print(f"pos_weight = {pos_w:.3f}  (n_neg/n_pos)")

# Functional weights
SUP_CAP = 2.0  # reduced: 5.0 caused all-suppressive collapse
def fw(n): return round(min(n_fc/(3*max(n,1)), SUP_CAP), 3)
w_unk, w_sup, w_act = fw(n_unk), fw(n_sup), fw(n_act)
print(f"func_weights = [{w_unk}, {w_sup}, {w_act}]")

CONFIG = {
    "hidden_dim"   : 128,
    "dropout"      : 0.1,
    "lr"           : 1e-4,  # fine-tune LR (warm start from val_auc=0.7959) training rate
    "weight_decay" : 1e-4,
    "batch_size"   : 64 if torch.cuda.is_available() else 16,
    "epochs"       : 60,   # fine-tune: 60 more epochs from checkpoint
    "patience"     : 20,
    "lambda1"      : 1.0,
    "lambda2"      : 0.10,       # 0.15 caused suppressive collapse (w_sup×λ₂=0.75 dominated BCE)
    "lambda3"      : 0.3,        # ranking loss kept
    "pos_weight"   : pos_w,
    "func_weights" : [w_unk, w_sup, w_act],
    "seed"         : 42,
    "version"      : "v0.7.0",
    "note"         : "Full integration: IEDB+TumorAgDB+TESLA+NCI/HiTIDE; scratch training",
    "datasets"     : "IEDB_v3+TumorAgDB1.0+TESLA_Wells2020+NCI_HiTIDE_Muller2023",
}
print("\nCONFIG v0.7:")
for k,v in CONFIG.items():
    print(f"  {k:<16}: {v}")

# ── Instantiate model ─────────────────────────────────────────────────────────
model = build_model(CONFIG).to(device)
# ── Warm start from best_model_v7.pt (val_auc=0.7959, epoch=44) ─────
# Searches for best_model_v7.pt first, then falls back to best_model.pt
ckpt_path = (find_file("best_model_v7.pt") or
             os.path.join(MODELS_DIR, "best_model_v7.pt") or
             find_file("best_model.pt") or
             os.path.join(MODELS_DIR, "best_model.pt"))
# Load metadata if available
_meta_path = (find_file("best_model_v7_meta.json") or
              os.path.join(MODELS_DIR, "best_model_v7_meta.json"))
if _meta_path and os.path.exists(_meta_path):
    import json as _jm
    with open(_meta_path) as _f: _meta = _jm.load(_f)
    print(f"  Checkpoint meta: val_auc={_meta.get('val_auc','?')}, epoch={_meta.get('epoch','?')}, version={_meta.get('version','?')}")
if os.path.exists(ckpt_path):
    try:
        raw = torch.load(ckpt_path, map_location=device)
        # Handle both formats:
        # Format A (raw): OrderedDict of weight tensors  <- what load_state_dict expects
        # Format B (wrapped): {"state_dict": ..., "val_auc": ..., "config": ...}
        sd = raw.get("state_dict", raw) if isinstance(raw, dict) and "state_dict" in raw else raw
        model.load_state_dict(sd)
        auc_str = f'  val_auc={raw["val_auc"]:.4f}' if isinstance(raw, dict) and "val_auc" in raw else ""
        print("Warm-start: loaded " + ckpt_path + auc_str)
    except Exception as e:
        print("Checkpoint incompatible (" + str(e) + ") — training from scratch")
else:
    print("No checkpoint — training from scratch")
n_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model ready: {n_p:,} trainable parameters")


Device check: cuda OK

Final distribution: pos=10,935 neg=6,813
Functional: unk=7,565 sup=479 act=9,704
pos_weight = 0.623  (n_neg/n_pos)
func_weights = [0.782, 2.0, 0.61]

CONFIG v0.7:
  hidden_dim      : 128
  dropout         : 0.1
  lr              : 0.0001
  weight_decay    : 0.0001
  batch_size      : 64
  epochs          : 60
  patience        : 20
  lambda1         : 1.0
  lambda2         : 0.1
  lambda3         : 0.3
  pos_weight      : 0.623
  func_weights    : [0.782, 2.0, 0.61]
  seed            : 42
  version         : v0.7.0
  note            : Full integration: IEDB+TumorAgDB+TESLA+NCI/HiTIDE; scratch training
  datasets        : IEDB_v3+TumorAgDB1.0+TESLA_Wells2020+NCI_HiTIDE_Muller2023
Checkpoint incompatible (Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to

## Cell 7 — Train v0.3
Fine-tunes from best_model.pt checkpoint (warm start) OR trains from scratch if not found.

In [10]:
import time, json
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

torch.manual_seed(CONFIG["seed"])

# ── Datasets ────────────────────────────────────────────────
print("Building graph datasets...")
train_ds = NeoantigenGraphDataset(train_combined)
val_ds   = NeoantigenGraphDataset(val_orig)
test_ds  = NeoantigenGraphDataset(test_orig)

tl  = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=0)
vl  = DataLoader(val_ds,   batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0)
tsl = DataLoader(test_ds,  batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0)

# v0.7: scratch training — no warm start
# Correct for combined dataset distribution
# ── Warm start: load best_model_v7.pt checkpoint ────────────────────────
_ws_path = (find_file("best_model_v7.pt") or
            os.path.join(MODELS_DIR, "best_model_v7.pt"))
if _ws_path and os.path.exists(_ws_path):
    _raw = torch.load(_ws_path, map_location=device, weights_only=False)
    _sd  = _raw.get("state_dict", _raw) if isinstance(_raw, dict) and "state_dict" in _raw else _raw
    model.load_state_dict(_sd)
    print(f"Warm start: loaded {_ws_path}")
    print(f"  Continuing from val_auc=0.7959 (epoch 44)")
    print(f"  Fine-tuning with Focal Loss + 8-epoch warmup + LR 5e-4")
else:
    print("WARNING: best_model_v7.pt not found — training from scratch")
    print("  Upload best_model_v7.pt to neetuaashi/met3dneoantigen-v7")

# ── Model check (instantiated in Cell 15; guard for partial re-runs) ─────────
if "model" not in dir() or model is None:
    print("WARNING: model not found — run Cell 12 + Cell 15 first, or re-running now...")
    model = build_model(CONFIG).to(device)
n_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {n_p:,}")

optimiser = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"],
                               weight_decay=CONFIG["weight_decay"])
# ── LR schedule: linear warmup (5 ep) then ReduceLROnPlateau ─────────────────
# Warmup prevents early GATConv attention collapse (seen as LR halving at ep 11
# in GPU run — model bouncing in sharp loss basin before attention stabilises).
WARMUP_EPOCHS = 3  # short warmup for fine-tuning (weights already oriented)
WARMUP_START_LR = 1e-5
_base_lr = CONFIG["lr"]

def warmup_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return WARMUP_START_LR/_base_lr + (1 - WARMUP_START_LR/_base_lr) * epoch/WARMUP_EPOCHS
    return 1.0  # hand off to ReduceLROnPlateau after warmup

warmup_sched = torch.optim.lr_scheduler.LambdaLR(optimiser, lr_lambda=warmup_lambda)
plateau_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimiser, mode="max", factor=0.5, patience=12, min_lr=1e-5)  # increased: Focal noisier than BCE

# ── Focal Loss (replaces BCE) ─────────────────────────────────────────────────
# Focal Loss = −α(1−p)^γ log(p). γ=2 down-weights easy/confident predictions,
# focusing gradient on hard uncertain examples — the minority immunogenic class.
# α=pos_weight normalises class imbalance (same role as BCE pos_weight).
FOCAL_GAMMA = 2.0
fw  = torch.tensor(CONFIG["func_weights"], dtype=torch.float).to(device)
ce  = nn.CrossEntropyLoss(weight=fw, ignore_index=-1)

def focal_loss(logit, y, gamma=FOCAL_GAMMA, alpha=None):
    bce_loss = nn.functional.binary_cross_entropy_with_logits(
        logit, y, reduction="none")
    p = torch.sigmoid(logit)
    p_t = p * y + (1 - p) * (1 - y)            # prob of correct class
    focal_weight = (1 - p_t) ** gamma            # down-weight easy examples
    if alpha is not None:
        alpha_t = alpha * y + (1 - alpha) * (1 - y)
        focal_weight = alpha_t * focal_weight
    return (focal_weight * bce_loss).mean()

_alpha = CONFIG["pos_weight"] / (1 + CONFIG["pos_weight"])  # convert to [0,1] range

def compute_loss(out, batch):
    logit = out["logit_immuno"].squeeze(-1)
    y_im  = batch.y_immuno.to(device).squeeze(-1).float()
    y_fn  = batch.y_func.to(device).squeeze(-1).long()
    sc    = out["score_activate"].squeeze(-1)
    l1 = focal_loss(logit, y_im, gamma=FOCAL_GAMMA, alpha=_alpha)
    mask = y_fn >= 0
    l2 = ce(out["logit_func"][mask], y_fn[mask]) if mask.sum()>0          else torch.tensor(0., device=device)
    pos=y_im==1; neg=y_im==0
    l3 = torch.clamp(1.-sc[pos].unsqueeze(1)+sc[neg].unsqueeze(0),min=0).mean()          if pos.sum()>0 and neg.sum()>0 else torch.tensor(0.,device=device)
    return CONFIG["lambda1"]*l1 + CONFIG["lambda2"]*l2 + CONFIG["lambda3"]*l3

def evaluate(loader):
    model.eval()
    pa,ya,fp,ft=[],[],[],[]
    tl=0.
    with torch.no_grad():
        for b in loader:
            b=b.to(device); out=model(b)
            tl+=compute_loss(out,b).item()
            pa.extend(torch.sigmoid(out["logit_immuno"].squeeze(-1)).cpu().numpy())
            fp.extend(out["logit_func"].argmax(1).cpu().numpy())
            ya.extend(b.y_immuno.squeeze(-1).cpu().numpy())
            ft.extend(b.y_func.squeeze(-1).cpu().numpy())
    pa=np.array(pa);ya=np.array(ya);fp=np.array(fp);ft=np.array(ft)
    try:
        from sklearn.metrics import roc_curve as _rc
        _fpr,_tpr,_thr = _rc(ya, pa)
        _j = _tpr - _fpr
        _opt = float(_thr[_j.argmax()]) if len(_thr) > 0 else 0.42
        _opt = max(0.1, min(0.9, _opt))
    except:
        _opt = 0.42
    pred = (pa >= _opt).astype(int)
    try: auc=roc_auc_score(ya,pa)
    except: auc=0.
    f1=f1_score(ya,pred,zero_division=0)
    acc=accuracy_score(ya,pred)
    lab=ft>=0
    facc=accuracy_score(ft[lab],fp[lab]) if lab.sum()>0 else 0.
    # Per-class functional accuracy
    fc_detail={}
    for fc,nm in [(0,"unknown"),(1,"suppressive"),(2,"activating")]:
        m=ft==fc
        fc_detail[nm]=round(accuracy_score(ft[m],fp[m]),4) if m.sum()>0 else None
    return {"loss":round(tl/len(loader),4),"auc":round(float(auc),4),
            "f1":round(float(f1),4),"acc":round(float(acc),4),
            "func_acc":round(float(facc),4),"fc_detail":fc_detail,
            "threshold":round(float(_opt),4)}

# ── Training ─────────────────────────────────────────────────
best_auc=0.; best_threshold=0.42; val_auc=0.; patience_count=0; history=[]; save_path=os.path.join(MODELS_DIR,"best_model_v7.pt")
print(f"\n{'Epoch':>5}  {'TrLoss':>8}  {'ValAUC':>7}  {'F1':>7}  {'FuncAcc':>8}  {'ActAcc':>7}  {'SupAcc':>7}  {'LR':>8}")
print("-"*76)

for epoch in range(1, CONFIG["epochs"]+1):
    t0=time.time(); model.train(); tr=0.
    for b in tl:
        b=b.to(device); optimiser.zero_grad()
        loss=compute_loss(model(b),b); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimiser.step(); tr+=loss.item()
    tr/=len(tl)
    vm=evaluate(vl)
    val_auc=vm["auc"]
    if epoch <= WARMUP_EPOCHS:
        warmup_sched.step()         # linear warmup
    else:
        plateau_sched.step(val_auc) # ReduceLROnPlateau after warmup
    act_acc=vm["fc_detail"].get("activating")
    sup_acc=vm["fc_detail"].get("suppressive")
    act_str=f"{act_acc:.4f}" if act_acc is not None else "  n/a "
    sup_str=f"{sup_acc:.4f}" if sup_acc is not None else "  n/a "
    cur_lr=optimiser.param_groups[0]["lr"]
    print(f"{epoch:>5}  {tr:>8.4f}  {vm['auc']:>7.4f}  {vm['f1']:>7.4f}  "
          f"{vm['func_acc']:>8.4f}  {act_str}  {sup_str}  {cur_lr:.1e}  ({time.time()-t0:.1f}s)")
    history.append({"epoch":epoch,"train_loss":round(tr,4),
                    **{f"val_{k}":v for k,v in vm.items() if k!="fc_detail"}})
    if vm["auc"]>best_auc:
        best_auc=vm["auc"]; patience_count=0
        best_threshold=vm.get("threshold", 0.42)  # save Youden-optimal threshold
        torch.save(model.state_dict(), save_path)
        import json as _json
        meta_path = save_path.replace(".pt", "_meta.json")
        with open(meta_path, "w") as _mf:
            _json.dump({"val_auc": float(best_auc), "epoch": epoch,
                        "version": CONFIG.get("version","v0.7")}, _mf)
        print(f"  ✓ Best AUC={best_auc:.4f} — saved")
    else:
        patience_count+=1
        if patience_count>=CONFIG["patience"]:
            print(f"\nEarly stopping at epoch {epoch}"); break

# ── Test evaluation ───────────────────────────────────────────
# Save training history for plotting cell
hist_data = {"history": history, "best_auc": best_auc, "best_threshold": best_threshold}
_hist_path = os.path.join(WORK_DIR, "training_history.json")
with open(_hist_path, "w") as _f:
    json.dump(hist_data, _f, default=str)
print(f"Training history saved: {_hist_path}")

print("\nLoading best checkpoint...")
raw=torch.load(save_path,map_location=device,weights_only=False)
sd=raw.get("state_dict",raw) if isinstance(raw,dict) and "state_dict" in raw else raw
model.load_state_dict(sd)
print(f"Checkpoint loaded (val_auc={best_auc:.4f}, threshold={best_threshold:.3f})")
test_metrics=evaluate(tsl)
# Override F1 using val-optimal threshold (more stable than recomputing Youden on test)
try:
    model.eval()
    _pa_test, _ya_test = [], []
    with torch.no_grad():
        for _b in tsl:
            _b = _b.to(device)
            _out = model(_b)
            _pa_test.extend(torch.sigmoid(_out["logit_immuno"].squeeze(-1)).cpu().numpy())
            _ya_test.extend(_b.y_immuno.squeeze(-1).cpu().numpy())
    import numpy as _np2
    from sklearn.metrics import f1_score as _f1s, accuracy_score as _accs
    _pa_t = _np2.array(_pa_test); _ya_t = _np2.array(_ya_test)
    _pred_val_thr = (_pa_t >= best_threshold).astype(int)
    _pred_05 = (_pa_t >= 0.5).astype(int)
    f1_val_thr = _f1s(_ya_t, _pred_val_thr, zero_division=0)
    f1_05 = _f1s(_ya_t, _pred_05, zero_division=0)
    print(f"Test F1 @ val-threshold ({best_threshold:.3f}): {f1_val_thr:.4f}")
    print(f"Test F1 @ 0.500: {f1_05:.4f}")
    test_metrics["f1"] = round(float(max(f1_val_thr, f1_05)), 4)
    test_metrics["f1_val_threshold"] = round(float(f1_val_thr), 4)
    test_metrics["f1_0.5"] = round(float(f1_05), 4)
    test_metrics["threshold_used"] = float(best_threshold)
except Exception as _te:
    print(f"Test threshold re-eval failed: {_te}")
print("\n"+"="*55)
print("TEST RESULTS — fine-tune vs previous best (val_auc=0.7959)")
print("="*55)
# Baselines: v0.2 (original), v0.7 (TumorAgDB1.0 only)
v2={"auc":0.7803,"f1":0.5678,"acc":0.7325,"func_acc":0.8759}  # v0.7+TumorAgDB2.0+Focal (previous best)
for k in ["auc","f1","acc","func_acc"]:
    v=test_metrics[k]; old=v2[k]
    delta=v-old; arrow="↑" if delta>0 else "↓"
    print(f"  {k:<12}: {v:.4f}  ({arrow}{abs(delta):.4f} vs v0.2 {old:.4f})")
print("\nPer-class functional accuracy:")
for cls,acc in test_metrics.get("fc_detail",{}).items():
    print(f"  {cls:<15}: {acc:.4f}" if acc else f"  {cls:<15}: N/A (no samples)")

hist_data={"history":history,"test":test_metrics,"config":CONFIG}
with open(os.path.join(WORK_DIR,"training_history_v7.json"),"w") as f:
    json.dump(hist_data,f,indent=2)
print(f"\nHistory saved: training_history_v3.json")

Building graph datasets...
  Loaded 17,748 samples
  Loaded 1,263 samples
  Loaded 1,260 samples
Warm start: loaded /kaggle/input/datasets/neetuaashi/met3dneoantigen-v7/best_model_v7.pt
  Continuing from val_auc=0.7959 (epoch 44)
  Fine-tuning with Focal Loss + 8-epoch warmup + LR 5e-4
Parameters: 262,277

Epoch    TrLoss   ValAUC       F1   FuncAcc   ActAcc   SupAcc        LR
----------------------------------------------------------------------------
    1    0.2609   0.7958   0.6002    0.8543  0.0000  0.0000  4.0e-05  (32.3s)
  ✓ Best AUC=0.7958 — saved


KeyboardInterrupt: 

## Cell 8 — Training Curves + v0.2 vs v0.3 Comparison

In [ ]:
import matplotlib.pyplot as plt
import os, json

# ── Load training history (from memory or file) ───────────────────────────────
if "hist_data" not in dir() or hist_data is None:
    _candidates = [
        os.path.join(MODELS_DIR, "training_history_v7.json"),
        os.path.join(MODELS_DIR, "training_history_v3.json"),
        os.path.join(WRK,        "training_history.json"),
        os.path.join(WRK,        "training_history_v3.json"),
    ]
    for _p in _candidates:
        if os.path.exists(_p):
            with open(_p) as _f:
                hist_data = json.load(_f)
            print(f"Loaded hist_data from: {_p}")
            break
    else:
        raise RuntimeError(
            "hist_data not found in memory or on disk.\n"
            "Re-run Cell 19 (training) to generate training_history_v7.json."
        )

history = hist_data["history"]
epochs  = [h["epoch"]          for h in history]
tr_loss = [h["train_loss"]     for h in history]
val_auc = [h["val_auc"]        for h in history]
val_f1  = [h["val_f1"]         for h in history]
val_fa  = [h.get("val_func_acc", 0) for h in history]

# ── Also pull SupAcc and ActAcc if saved ─────────────────────────────────────
val_supacc = [h.get("val_suppressive_acc", h.get("val_SupAcc", None)) for h in history]
val_actacc = [h.get("val_activating_acc",  h.get("val_ActAcc", None)) for h in history]
has_supacc = any(v is not None for v in val_supacc)

# ── Plots ─────────────────────────────────────────────────────────────────────
ncols = 5 if has_supacc else 4
fig, axes = plt.subplots(1, ncols, figsize=(5*ncols, 4))

axes[0].plot(epochs, tr_loss, color="#185FA5", lw=2, label="v0.7")
axes[0].axhline(1.3356, ls="--", color="gray", alpha=0.5, label="v0.2 baseline")
axes[0].set_title("Training loss"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, val_auc, color="#1D9E75", lw=2, label="v0.7 val AUROC")
axes[1].axhline(0.7562, ls="--", color="gray", alpha=0.5, label="v0.2 AUROC=0.756")
axes[1].axhline(0.878,  ls=":",  color="#D85A30", alpha=0.6, label="v0.8 target=0.878")
axes[1].set_ylim(0.5, 1.0); axes[1].set_title("Validation AUROC")
axes[1].legend(fontsize=7); axes[1].grid(alpha=0.3)

axes[2].plot(epochs, val_f1, color="#7F77DD", lw=2, label="v0.7 F1")
axes[2].axhline(0.5203, ls="--", color="gray", alpha=0.5, label="v0.2 F1=0.520")
axes[2].set_ylim(0, 1); axes[2].set_title("Validation F1")
axes[2].legend(); axes[2].grid(alpha=0.3)

axes[3].plot(epochs, val_fa, color="#D85A30", lw=2, label="v0.7 func acc")
axes[3].axhline(0.9733, ls="--", color="gray", alpha=0.5, label="v0.2=0.973 (artefact)")
axes[3].axhline(0.333,  ls=":",  color="silver", alpha=0.5, label="random baseline")
axes[3].set_ylim(0, 1.05); axes[3].set_title("Functional class accuracy")
axes[3].legend(fontsize=7); axes[3].grid(alpha=0.3)

if has_supacc:
    sup_vals = [v if v is not None else 0 for v in val_supacc]
    act_vals = [v if v is not None else 0 for v in val_actacc]
    axes[4].plot(epochs, sup_vals, color="#639922", lw=2, label="SupAcc")
    axes[4].plot(epochs, act_vals, color="#BA7517", lw=1.5, ls="--", label="ActAcc")
    axes[4].axhline(0.0, ls=":", color="silver", alpha=0.4)
    axes[4].set_ylim(0, 1.0); axes[4].set_title("Functional recall (Sup / Act)")
    axes[4].legend(fontsize=8); axes[4].grid(alpha=0.3)

for ax in axes:
    ax.set_xlabel("Epoch")

plt.suptitle(
    f"Met-3DNet-VI v0.7 — Training curves  (best val AUROC={max(val_auc):.4f})",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
out_png = os.path.join(WORK_DIR, "training_curves_v7.png")
fig.savefig(out_png, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_png}")

# ── Final summary ─────────────────────────────────────────────────────────────
print("\n" + "="*55)
print("TRAINING SUMMARY — Met-3DNet-VI v0.7")
print("="*55)
print(f"  Epochs run          : {len(history)}")
print(f"  Best val AUROC      : {max(val_auc):.4f}  (epoch {val_auc.index(max(val_auc))+1})")
print(f"  Best val F1         : {max(val_f1):.4f}")
print(f"  Final train loss    : {tr_loss[-1]:.4f}")
if has_supacc:
    non_zero_sup = [v for v in sup_vals if v > 0]
    print(f"  SupAcc peak         : {max(sup_vals):.4f}  (epoch {sup_vals.index(max(sup_vals))+1})")
    print(f"  SupAcc mean (non-0) : {sum(non_zero_sup)/len(non_zero_sup):.4f}  ({len(non_zero_sup)}/{len(sup_vals)} epochs)")
    print(f"  ActAcc peak         : {max(act_vals):.4f}")
if "test" in hist_data:
    tm = hist_data["test"]
    print(f"\n  Test AUROC          : {tm.get('auc','N/A')}")
    print(f"  Test F1             : {tm.get('f1','N/A')}")
    print(f"  Test accuracy       : {tm.get('acc','N/A')}")
    if "fc_detail" in tm:
        print("  Per-class recall:")
        for cls, acc in tm["fc_detail"].items():
            print(f"    {cls:<15}: {acc:.4f}" if acc is not None else f"    {cls:<15}: N/A")
print(f"\n  Download: best_model_v7.pt + training_history_v7.json")
print(f"  Add to Kaggle dataset: neetuaashi/met3dneoantigen-v7")
